# NRC-Cal: Neural Regression Collapse and Probabilistic Calibratability

This is the single Colab notebook for the lab experiment proposed in `NRC_Calibration_Paper_Proposal.md`. The scientific question is:

> For a frozen regression model, does the distance from Neural Regression Collapse (NRC) geometry predict probabilistic miscalibration, and can that distance drive a post-hoc correction without retraining?

The notebook runs a reproducible **8-dataset pilot** end to end. It trains small Gaussian MLPs only to create frozen checkpoints when the external QRT checkpoints are not present, then performs one-pass feature extraction, NRC measurement, BASE/QR/NRC-Cal evaluation, correlation testing, plots, ablations, and export. This pilot is a lab validation and debugging run, not the final 57-dataset paper result.

The full-scale QRT-57 analysis is included as an optional artifact path. It must use the exact QRT calibration split and genuine BASE Gaussian checkpoints; cloning the upstream repositories alone does not provide those checkpoints.


## Experimental contract and decision rule

The proposal requires these invariants:

- Use the calibration split for fitting and the disjoint test split for final metrics.
- Start with a single Gaussian head (`K=1`), because the published NRC theory is clearest there.
- Report PCE as the primary metric; NLL, CRPS, coverage, and sharpness are secondary.
- Correlate dataset-level NRC distance with BASE PCE and with residual PCE after standard quantile recalibration (QR).
- Do not call the result a positive finding until the prespecified rule is applied: `abs(Spearman rho) >= 0.5 and p < 0.1` is GO; a visible but weaker trend is an EXTEND decision; otherwise the pilot is NO-GO.

Published NRC1-NRC3 are implemented from the NeurIPS 2024 definitions. `D_dataset`, sample distances, and the log-Mahalanobis covariance map are new NRC-Cal proposals and are labelled as such throughout.


In [ ]:
# 0. Colab setup: optionally persist all generated outputs in Google Drive.
from pathlib import Path
import os, sys, random, json, subprocess, platform

MOUNT_DRIVE = True
PROJECT_NAME = "NRC_CALIB_CODE"
try:
    from google.colab import drive
    if MOUNT_DRIVE:
        drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

if MOUNT_DRIVE and Path("/content/drive/MyDrive").is_dir():
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_ROOT = Path("/content") / PROJECT_NAME
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ("src", "configs", "docs", "external", "outputs", "figures", "checkpoints"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
os.environ["NRC_CAL_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["NRC_CAL_CHECKPOINT_ROOT"] = str(PROJECT_ROOT / "checkpoints")
SEED = 2026
random.seed(SEED)
print(f"PROJECT_ROOT={PROJECT_ROOT.resolve()}")
print(f"Python={platform.python_version()}")


## 1. Install the Colab runtime

The install uses notebook shell commands (`!pip`) as requested. It avoids the upstream repositories' old pinned PyTorch versions so Colab's CUDA runtime remains usable.


In [ ]:
# Kaggle Notebook runtime dependencies. The shell command is intentionally visible/re-runnable.
!pip install -q numpy scipy pandas scikit-learn matplotlib seaborn tqdm psutil openml "kagglehub[pandas-datasets]" pyarrow openpyxl
import numpy, scipy, pandas, sklearn, torch, matplotlib
print({"numpy": numpy.__version__, "scipy": scipy.__version__, "pandas": pandas.__version__, "torch": torch.__version__})
print("CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


## 2. Clone the cited external repositories (optional but reproducible)

These repositories provide the QRT methodology, dataset loaders, and comparison baselines. Their public git trees do **not** contain the pretrained BASE checkpoint zoo required by the proposal, so this cell records that fact rather than silently claiming success.


In [ ]:
# Clone with !git so the commands are visible and reproducible in Colab.
!mkdir -p "$NRC_CAL_SHELL_ROOT/external"
!if [ ! -d $NRC_CAL_SHELL_ROOT/external/quantile-recalibration-training/.git ]; then git clone --depth 1 https://github.com/Vekteur/quantile-recalibration-training.git $NRC_CAL_SHELL_ROOT/external/quantile-recalibration-training; fi
!if [ ! -d $NRC_CAL_SHELL_ROOT/external/probabilistic-calibration-study/.git ]; then git clone --depth 1 https://github.com/Vekteur/probabilistic-calibration-study.git $NRC_CAL_SHELL_ROOT/external/probabilistic-calibration-study; fi

from pathlib import Path
external_root = PROJECT_ROOT / "external"
for repo in ("quantile-recalibration-training", "probabilistic-calibration-study"):
    repo_path = external_root / repo
    checkpoints = [p for p in repo_path.rglob("*") if p.suffix.lower() in {".ckpt", ".pt", ".pth", ".safetensors"}]
    print(repo, "checkpoint files:", len(checkpoints))
print("If this reports zero, continue with the self-contained pilot below or mount genuine BASE checkpoints under PROJECT_ROOT/checkpoints.")


## 3. Embed the NRC-Cal implementation

The uploaded notebook carries the reusable implementation inside itself. It writes the package at runtime, so no repository checkout is needed to import the geometry, prediction, calibration, metrics, or plotting code.


In [ ]:
# Write the embedded source tree. The payload is generated into this notebook at build time.
import base64, importlib, types
SOURCE_FILES = {"src/calibration/__init__.py": "IiIiUHJvcG9zZWQgTlJDLUNhbCBhbmQgYmFzZWxpbmUgcG9zdC1ob2MgY2FsaWJyYXRpb24gaGVscGVycy4iIiIK", "src/calibration/baselines.py": "IiIiVHJhbnNwYXJlbnQgcG9zdC1ob2MgYmFzZWxpbmUgYWRhcHRlcnMgZm9yIFFSVCBjb21wYXJpc29ucy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5pbnRlcnBvbGF0ZSBpbXBvcnQgUGNoaXBJbnRlcnBvbGF0b3IKZnJvbSBza2xlYXJuLmlzb3RvbmljIGltcG9ydCBJc290b25pY1JlZ3Jlc3Npb24KCmZyb20gbW9kZWxzLnByZWRpY3Rpb25zIGltcG9ydCBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUXVhbnRpbGVSZWNhbGlicmF0b3I6CiAgICAiIiJNb25vdG9uZSBQSVQtdG8tUElUIG1hcCBmaXR0ZWQgb24gYSBjYWxpYnJhdGlvbiBzcGxpdC4iIiIKCiAgICBzb3VyY2U6IG5wLm5kYXJyYXkKICAgIHRhcmdldDogbnAubmRhcnJheQoKICAgIGRlZiBtYXAoc2VsZiwgcGl0OiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgICAgICIiIk1hcCBQSVQgdmFsdWVzIHRocm91Z2ggdGhlIGZpdHRlZCBtb25vdG9uZSBpbnRlcnBvbGF0aW9uLiIiIgogICAgICAgIHJldHVybiBucC5jbGlwKFBjaGlwSW50ZXJwb2xhdG9yKHNlbGYuc291cmNlLCBzZWxmLnRhcmdldCwgZXh0cmFwb2xhdGU9VHJ1ZSkocGl0KSwgMC4wLCAxLjApCgoKZGVmIGZpdF9xdWFudGlsZV9yZWNhbGlicmF0b3IocHJlZGljdGlvbjogR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbiwgdGFyZ2V0czogbnAubmRhcnJheSwgbGV2ZWxzOiBpbnQgPSAxMDApIC0+IFF1YW50aWxlUmVjYWxpYnJhdG9yOgogICAgIiIiRml0IHRoZSBzdGFuZGFyZCBlbXBpcmljYWwgcXVhbnRpbGUtcmVjYWxpYnJhdGlvbiBtYXAgKFFSIGJhc2VsaW5lKS4iIiIKICAgIHBpdCA9IHByZWRpY3Rpb24uY2RmXzFkKHRhcmdldHMpCiAgICBzb3VyY2UgPSBucC5saW5zcGFjZSgwLjAsIDEuMCwgbGV2ZWxzICsgMilbMTotMV0KICAgIGVtcGlyaWNhbCA9IG5wLmFycmF5KFsocGl0IDw9IGxldmVsKS5tZWFuKCkgZm9yIGxldmVsIGluIHNvdXJjZV0pCiAgICBpc290b25pYyA9IElzb3RvbmljUmVncmVzc2lvbih5X21pbj0wLjAsIHlfbWF4PTEuMCwgaW5jcmVhc2luZz1UcnVlKS5maXQoc291cmNlLCBlbXBpcmljYWwpCiAgICB0YXJnZXQgPSBucC5jbGlwKGlzb3RvbmljLnByZWRpY3Qoc291cmNlKSwgMC4wLCAxLjApCiAgICByZXR1cm4gUXVhbnRpbGVSZWNhbGlicmF0b3IobnAucl9bMC4wLCBzb3VyY2UsIDEuMF0sIG5wLnJfWzAuMCwgdGFyZ2V0LCAxLjBdKQoKCmRlZiBhcHBseV9waXRfbWFwKHByZWRpY3Rpb246IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb24sIHRhcmdldHM6IG5wLm5kYXJyYXksIGNhbGlicmF0b3I6IFF1YW50aWxlUmVjYWxpYnJhdG9yKSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJuIHJlY2FsaWJyYXRlZCBQSVQgdmFsdWVzOyBmdWxsIGRpc3RyaWJ1dGlvbiBpbnZlcnNpb24gaXMgZGF0YS1kZXBlbmRlbnQuIiIiCiAgICByZXR1cm4gY2FsaWJyYXRvci5tYXAocHJlZGljdGlvbi5jZGZfMWQodGFyZ2V0cykpCgoKZGVmIGJhc2VsaW5lX3JlZ2lzdHJ5KCkgLT4gZGljdFtzdHIsIHN0cl06CiAgICAiIiJOYW1lIHRoZSByZXF1ZXN0ZWQgYmFzZWxpbmVzIGFuZCB0aGVpciBzb3VyY2UgcHJvdmVuYW5jZS4iIiIKICAgIHJldHVybiB7IlFSIjogImVtcGlyaWNhbCBQSVQgcXVhbnRpbGUgcmVjYWxpYnJhdGlvbiIsICJRUkMiOiAidXBzdHJlYW0gUVJUIHBvc3QtaG9jIHJlY2FsaWJyYXRpb24iLCAiUVJUQyI6ICJ1cHN0cmVhbSBRUlQgY2FsaWJyYXRpb24gdHJhaW5pbmciLCAiUVJFR0MiOiAidXBzdHJlYW0gUVJUIHJlZ3VsYXJpemVkIGNhbGlicmF0aW9uIn0K", "src/calibration/nrc_cal.py": "IiIiTlJDLUNhbCdzIGV4cGxpY2l0bHkgcHJvcG9zZWQgY2xvc2VkLWZvcm0gZnJvemVuLW1vZGVsIHNjYWxlIGNvcnJlY3Rpb24uIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkuc3BlY2lhbCBpbXBvcnQgZGlnYW1tYQoKZnJvbSBtb2RlbHMucHJlZGljdGlvbnMgaW1wb3J0IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb24KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBOUkNDYWxpYnJhdG9yOgogICAgIiIiRml0dGVkIGNvZWZmaWNpZW50cyBmb3IgdGhlIHByb3Bvc2VkIGxvZy1NYWhhbGFub2JpcyBzY2FsZSBtYXAuIiIiCgogICAgaW50ZXJjZXB0OiBmbG9hdAogICAgZ2VvbWV0cnlfc2xvcGU6IGZsb2F0CiAgICBkaXN0YW5jZV9tZWFuOiBmbG9hdAogICAgZGlzdGFuY2Vfc3RkOiBmbG9hdAogICAgdGFyZ2V0X2RpbWVuc2lvbjogaW50CiAgICByaWRnZTogZmxvYXQKICAgIHNjYWxlX21pbjogZmxvYXQKICAgIHNjYWxlX21heDogZmxvYXQKCiAgICBkZWYgc2NhbGVzKHNlbGYsIGRpc3RhbmNlczogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJDb21wdXRlIGJvdW5kZWQgcG9zaXRpdmUgTlJDLUNhbCBzY2FsZSBmYWN0b3JzIGZvciBuZXcgZXhhbXBsZXMuIiIiCiAgICAgICAgdmFsdWVzID0gbnAuYXNhcnJheShkaXN0YW5jZXMsIGR0eXBlPW5wLmZsb2F0NjQpLnJlc2hhcGUoLTEpCiAgICAgICAgeiA9ICh2YWx1ZXMgLSBzZWxmLmRpc3RhbmNlX21lYW4pIC8gbWF4KHNlbGYuZGlzdGFuY2Vfc3RkLCBucC5maW5mbyhmbG9hdCkuZXBzKQogICAgICAgIHRhdSA9IGZsb2F0KGRpZ2FtbWEoc2VsZi50YXJnZXRfZGltZW5zaW9uIC8gMi4wKSArIG5wLmxvZygyLjApIC0gbnAubG9nKHNlbGYudGFyZ2V0X2RpbWVuc2lvbikpCiAgICAgICAgcmF3ID0gbnAuZXhwKDAuNSAqIChzZWxmLmludGVyY2VwdCArIHNlbGYuZ2VvbWV0cnlfc2xvcGUgKiB6IC0gdGF1KSkKICAgICAgICByZXR1cm4gbnAuY2xpcChyYXcsIHNlbGYuc2NhbGVfbWluLCBzZWxmLnNjYWxlX21heCkKCiAgICBkZWYgdHJhbnNmb3JtKHNlbGYsIHByZWRpY3Rpb246IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb24sIGRpc3RhbmNlczogbnAubmRhcnJheSkgLT4gR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbjoKICAgICAgICAiIiJBcHBseSB0aGUgdmFsaWQgY292YXJpYW5jZS1wcmVzZXJ2aW5nIEdhdXNzaWFuLW1peHR1cmUgY29ycmVjdGlvbi4iIiIKICAgICAgICBzY2FsZSA9IHNlbGYuc2NhbGVzKGRpc3RhbmNlcykKICAgICAgICBpZiBzY2FsZS5zaGFwZVswXSAhPSBwcmVkaWN0aW9uLm1lYW5zLnNoYXBlWzBdOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbmUgTlJDIGRpc3RhbmNlIGlzIHJlcXVpcmVkIHBlciBwcmVkaWN0aW9uIikKICAgICAgICBjZW50ZXIgPSBwcmVkaWN0aW9uLm1lYW5bOiwgTm9uZSwgOl0KICAgICAgICBtZWFucyA9IGNlbnRlciArIHNjYWxlWzosIE5vbmUsIE5vbmVdICogKHByZWRpY3Rpb24ubWVhbnMgLSBjZW50ZXIpCiAgICAgICAgY292YXJpYW5jZXMgPSBzY2FsZVs6LCBOb25lLCBOb25lLCBOb25lXSAqKiAyICogcHJlZGljdGlvbi5jb3ZhcmlhbmNlcwogICAgICAgIHJldHVybiBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uKHByZWRpY3Rpb24ud2VpZ2h0cy5jb3B5KCksIG1lYW5zLCBjb3ZhcmlhbmNlcykKCgpkZWYgbWFoYWxhbm9iaXNfc2NhbGVfcmVzaWR1YWxzKHByZWRpY3Rpb246IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb24sIHRhcmdldHM6IG5wLm5kYXJyYXksIGppdHRlcjogZmxvYXQgPSAxZS04KSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJuIHByb3Bvc2VkIGBxX2lgLCB1c2luZyBtaXh0dXJlIHRvdGFsIGNvdmFyaWFuY2UgYW5kIENob2xlc2t5IHNvbHZlcy4iIiIKICAgIHkgPSBucC5hc2FycmF5KHRhcmdldHMsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBtZWFuLCBjb3ZhcmlhbmNlID0gcHJlZGljdGlvbi5tZWFuLCBwcmVkaWN0aW9uLnRvdGFsX2NvdmFyaWFuY2UKICAgIGlmIHkuc2hhcGUgIT0gbWVhbi5zaGFwZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0YXJnZXRzIG11c3QgbWF0Y2ggcHJlZGljdGlvbiBtZWFucyIpCiAgICB2YWx1ZXMgPSBucC5lbXB0eSh5LnNoYXBlWzBdLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgZm9yIGluZGV4LCAocmVzaWR1YWwsIG1hdHJpeCkgaW4gZW51bWVyYXRlKHppcCh5IC0gbWVhbiwgY292YXJpYW5jZSwgc3RyaWN0PVRydWUpKToKICAgICAgICBzdGFibGUgPSBtYXRyaXggKyBqaXR0ZXIgKiBucC5leWUobWF0cml4LnNoYXBlWzBdKQogICAgICAgIHNvbHV0aW9uID0gbnAubGluYWxnLnNvbHZlKHN0YWJsZSwgcmVzaWR1YWwpCiAgICAgICAgdmFsdWVzW2luZGV4XSA9IHJlc2lkdWFsIEAgc29sdXRpb24gLyByZXNpZHVhbC5zaGFwZVswXQogICAgcmV0dXJuIG5wLm1heGltdW0odmFsdWVzLCBqaXR0ZXIpCgoKZGVmIGZpdF9ucmNfY2FsaWJyYXRvcihwcmVkaWN0aW9uOiBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uLCB0YXJnZXRzOiBucC5uZGFycmF5LCBkaXN0YW5jZXM6IG5wLm5kYXJyYXksICosIHJpZGdlOiBmbG9hdCA9IDFlLTYsIHNjYWxlX21pbjogZmxvYXQgPSAwLjI1LCBzY2FsZV9tYXg6IGZsb2F0ID0gNC4wLCBqaXR0ZXI6IGZsb2F0ID0gMWUtOCkgLT4gTlJDQ2FsaWJyYXRvcjoKICAgICIiIkZpdCBOUkMtQ2FsIGJ5IHRoZSBkb2N1bWVudGVkIGNsb3NlZC1mb3JtIHJpZGdlIHJlZ3Jlc3Npb24gb24gY2FsaWJyYXRpb24gZGF0YS4iIiIKICAgIGlmIHJpZGdlIDwgMCBvciBub3QgKDAgPCBzY2FsZV9taW4gPD0gc2NhbGVfbWF4KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJSZXF1aXJlIHJpZGdlID49IDAgYW5kIDAgPCBzY2FsZV9taW4gPD0gc2NhbGVfbWF4IikKICAgIGQgPSBucC5hc2FycmF5KGRpc3RhbmNlcywgZHR5cGU9bnAuZmxvYXQ2NCkucmVzaGFwZSgtMSkKICAgIGlmIGQuc2hhcGVbMF0gIT0gcHJlZGljdGlvbi5tZWFucy5zaGFwZVswXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbmUgTlJDIGRpc3RhbmNlIGlzIHJlcXVpcmVkIGZvciBlYWNoIGNhbGlicmF0aW9uIHRhcmdldCIpCiAgICBtZWFuLCBzdGQgPSBmbG9hdChkLm1lYW4oKSksIGZsb2F0KGQuc3RkKCkpCiAgICB6ID0gKGQgLSBtZWFuKSAvIG1heChzdGQsIGppdHRlcikKICAgIGRlc2lnbiA9IG5wLmNvbHVtbl9zdGFjaygobnAub25lc19saWtlKHopLCB6KSkKICAgIHBlbmFsdHkgPSBucC5kaWFnKCgwLjAsIHJpZGdlKSkKICAgIHJlc3BvbnNlID0gbnAubG9nKG1haGFsYW5vYmlzX3NjYWxlX3Jlc2lkdWFscyhwcmVkaWN0aW9uLCB0YXJnZXRzLCBqaXR0ZXIpKQogICAgY29lZmZpY2llbnRzID0gbnAubGluYWxnLnNvbHZlKGRlc2lnbi5UIEAgZGVzaWduICsgcGVuYWx0eSwgZGVzaWduLlQgQCByZXNwb25zZSkKICAgIHJldHVybiBOUkNDYWxpYnJhdG9yKGZsb2F0KGNvZWZmaWNpZW50c1swXSksIGZsb2F0KGNvZWZmaWNpZW50c1sxXSksIG1lYW4sIHN0ZCwgcHJlZGljdGlvbi5tZWFucy5zaGFwZVsyXSwgcmlkZ2UsIHNjYWxlX21pbiwgc2NhbGVfbWF4KQoKCmRlZiBzZWxlY3RfcmlkZ2UocHJlZGljdGlvbjogR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbiwgdGFyZ2V0czogbnAubmRhcnJheSwgZGlzdGFuY2VzOiBucC5uZGFycmF5LCBjYW5kaWRhdGVzOiB0dXBsZVtmbG9hdCwgLi4uXSA9ICgwLjAsIDFlLTgsIDFlLTYsIDFlLTQsIDFlLTIpKSAtPiBOUkNDYWxpYnJhdG9yOgogICAgIiIiU2VsZWN0IGEgZGV0ZXJtaW5pc3RpYyBjYWxpYnJhdGlvbi1vbmx5IHJpZGdlIHZhbHVlIGJ5IHVuaXZhcmlhdGUgUENFIG9yIE5MTC4iIiIKICAgIGZyb20gbWV0cmljcy5ldmFsdWF0aW9uIGltcG9ydCBuZWdhdGl2ZV9sb2dfbGlrZWxpaG9vZCwgcHJvYmFiaWxpdHlfY2FsaWJyYXRpb25fZXJyb3IKCiAgICBiZXN0OiB0dXBsZVtmbG9hdCwgTlJDQ2FsaWJyYXRvcl0gfCBOb25lID0gTm9uZQogICAgZm9yIHJpZGdlIGluIGNhbmRpZGF0ZXM6CiAgICAgICAgZml0dGVkID0gZml0X25yY19jYWxpYnJhdG9yKHByZWRpY3Rpb24sIHRhcmdldHMsIGRpc3RhbmNlcywgcmlkZ2U9cmlkZ2UpCiAgICAgICAgY2FsaWJyYXRlZCA9IGZpdHRlZC50cmFuc2Zvcm0ocHJlZGljdGlvbiwgZGlzdGFuY2VzKQogICAgICAgIHNjb3JlID0gcHJvYmFiaWxpdHlfY2FsaWJyYXRpb25fZXJyb3IoY2FsaWJyYXRlZCwgdGFyZ2V0cykgaWYgdGFyZ2V0cy5zaGFwZVsxXSA9PSAxIGVsc2UgbmVnYXRpdmVfbG9nX2xpa2VsaWhvb2QoY2FsaWJyYXRlZCwgdGFyZ2V0cykKICAgICAgICBpZiBiZXN0IGlzIE5vbmUgb3Igc2NvcmUgPCBiZXN0WzBdOgogICAgICAgICAgICBiZXN0ID0gKGZsb2F0KHNjb3JlKSwgZml0dGVkKQogICAgaWYgYmVzdCBpcyBOb25lOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm8gcmlkZ2UgY2FuZGlkYXRlcyBzdXBwbGllZCIpCiAgICByZXR1cm4gYmVzdFsxXQo=", "src/datasets/__init__.py": "IiIiRGF0YXNldCBtYW5pZmVzdHMgYW5kIHNwbGl0IHZhbGlkYXRpb24uIiIiCg==", "src/datasets/manifest.py": "IiIiVGhlIGV4YWN0IDU3IHRhYnVsYXIgZGF0YXNldHMgY29uZmlndXJlZCBieSB0aGUgUVJUIHVwc3RyZWFtIHJlcG9zaXRvcnkuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIERhdGFzZXRTcGVjOgogICAgIiIiVXBzdHJlYW0gZGF0YXNldCBpZGVudGlmaWVyIGFuZCBzb3VyY2UgZ3JvdXAuIiIiCgogICAgbmFtZTogc3RyCiAgICBncm91cDogc3RyCgoKX0dST1VQUzogZGljdFtzdHIsIHR1cGxlW3N0ciwgLi4uXV0gPSB7CiAgICAidWNpIjogKCJDUFUiLCAiWWFjaHQiLCAiTVBHIiwgIkVuZXJneSIsICJDcmltZSIsICJGaXNoIiwgIkNvbmNyZXRlIiwgIkFpcmZvaWwiLCAiS2luOG5tIiwgIlBvd2VyIiwgIk5hdmFsIiwgIlByb3RlaW4iKSwKICAgICJvbWxfMjk3IjogKCJ3aW5lX3F1YWxpdHkiLCAiaXNvbGV0IiwgImNwdV9hY3QiLCAic3VsZnVyIiwgIkJyYXppbGlhbl9ob3VzZXMiLCAiQWlsZXJvbnMiLCAiTWlhbWlIb3VzaW5nMjAxNiIsICJwb2wiLCAiZWxldmF0b3JzIiwgIkJpa2VfU2hhcmluZ19EZW1hbmQiLCAiZmlmYSIsICJjYWxpZm9ybmlhIiwgInN1cGVyY29uZHVjdCIsICJob3VzZV9zYWxlcyIsICJob3VzZV8xNkgiLCAiZGlhbW9uZHMiLCAibWVkaWNhbF9jaGFyZ2VzIiwgInllYXIiLCAibnljLXRheGktZ3JlZW4tZGVjLTIwMTYiKSwKICAgICJvbWxfMjk5IjogKCJhbmFsY2F0ZGF0YV9zdXByZW1lIiwgIk1lcmNlZGVzX0JlbnpfR3JlZW5lcl9NYW51ZmFjdHVyaW5nIiwgInZpc3VhbGl6aW5nX3NvaWwiLCAieXByb3BfNF8xIiwgIk9ubGluZU5ld3NQb3B1bGFyaXR5IiwgImJsYWNrX2ZyaWRheSIsICJTR0VNTV9HUFVfa2VybmVsX3BlcmZvcm1hbmNlIiwgInBhcnRpY3VsYXRlLW1hdHRlci11a2Fpci0yMDE3IiksCiAgICAib21sXzI2OSI6ICgidGVjYXRvciIsICJib3N0b24iLCAiTUlQLTIwMTYtcmVncmVzc2lvbiIsICJzb2Ntb2IiLCAiTW9uZXliYWxsIiwgImhvdXNlX3ByaWNlc19ub21pbmFsIiwgInVzX2NyaW1lIiwgInF1YWtlIiwgInNwYWNlX2dhIiwgImFiYWxvbmUiLCAiU0FUMTEtSEFORC1ydW50aW1lLXJlZ3Jlc3Npb24iLCAiU2FudGFuZGVyX3RyYW5zYWN0aW9uX3ZhbHVlIiwgImNvbGxlZ2VzIiwgInRvcG9fMl8xIiwgIkFsbHN0YXRlX0NsYWltc19TZXZlcml0eSIsICJZb2xhbmRhIiwgIkJ1enppbnNvY2lhbG1lZGlhX1R3aXR0ZXIiLCAiQWlybGluZXNfRGVwRGVsYXlfMTBNIiksCn0KCgpkZWYgcXJ0NTdfbWFuaWZlc3QoKSAtPiB0dXBsZVtEYXRhc2V0U3BlYywgLi4uXToKICAgICIiIlJldHVybiB0aGUgNTcgbm9uLXRveSBRUlQgZGF0YXNldHMgaW4gdXBzdHJlYW0tY29uZmlnIG9yZGVyLiIiIgogICAgbWFuaWZlc3QgPSB0dXBsZShEYXRhc2V0U3BlYyhuYW1lLCBncm91cCkgZm9yIGdyb3VwLCBuYW1lcyBpbiBfR1JPVVBTLml0ZW1zKCkgZm9yIG5hbWUgaW4gbmFtZXMpCiAgICBpZiBsZW4obWFuaWZlc3QpICE9IDU3OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkV4cGVjdGVkIDU3IFFSVCBkYXRhc2V0cywgZm91bmQge2xlbihtYW5pZmVzdCl9IikKICAgIHJldHVybiBtYW5pZmVzdAo=", "src/datasets/splits.py": "IiIiU3BsaXQgZGlzY292ZXJ5IGFuZCBkZXRlcm1pbmlzdGljIGZhbGxiYWNrIHNwbGl0dGluZyBmb3IgdGFidWxhciBhcnJheXMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgTWFwcGluZwoKaW1wb3J0IG51bXB5IGFzIG5wCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVGFidWxhclNwbGl0czoKICAgICIiIk5vbi1vdmVybGFwcGluZyB0cmFpbiwgdmFsaWRhdGlvbiwgY2FsaWJyYXRpb24sIGFuZCB0ZXN0IGFycmF5cy4iIiIKCiAgICB0cmFpbl94OiBucC5uZGFycmF5CiAgICB0cmFpbl95OiBucC5uZGFycmF5CiAgICB2YWxpZGF0aW9uX3g6IG5wLm5kYXJyYXkKICAgIHZhbGlkYXRpb25feTogbnAubmRhcnJheQogICAgY2FsaWJyYXRpb25feDogbnAubmRhcnJheQogICAgY2FsaWJyYXRpb25feTogbnAubmRhcnJheQogICAgdGVzdF94OiBucC5uZGFycmF5CiAgICB0ZXN0X3k6IG5wLm5kYXJyYXkKCgpkZWYgc3BsaXRfdGFidWxhcih4OiBucC5uZGFycmF5LCB5OiBucC5uZGFycmF5LCBzZWVkOiBpbnQgPSAwLCByYXRpb3M6IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXQsIGZsb2F0XSA9ICgwLjY1LCAwLjEwLCAwLjE1LCAwLjEwKSkgLT4gVGFidWxhclNwbGl0czoKICAgICIiIkNyZWF0ZSByZXByb2R1Y2libGUgdHJhaW4vdmFsaWRhdGlvbi9jYWxpYnJhdGlvbi90ZXN0IHNwbGl0cyB3aXRob3V0IGxlYWthZ2UuIiIiCiAgICBpZiB4LnNoYXBlWzBdICE9IHkuc2hhcGVbMF0gb3IgeC5zaGFwZVswXSA8IDg6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigieCBhbmQgeSBtdXN0IGhhdmUgdGhlIHNhbWUgbGVuZ3RoIG9mIGF0IGxlYXN0IGVpZ2h0IikKICAgIGlmIG5vdCBucC5pc2Nsb3NlKHN1bShyYXRpb3MpLCAxLjApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNwbGl0IHJhdGlvcyBtdXN0IHN1bSB0byBvbmUiKQogICAgaW5kaWNlcyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKS5wZXJtdXRhdGlvbih4LnNoYXBlWzBdKQogICAgZW5kcyA9IG5wLmN1bXN1bShucC5hc2FycmF5KHJhdGlvc1s6LTFdKSAqIHguc2hhcGVbMF0pLmFzdHlwZShpbnQpCiAgICBncm91cHMgPSBucC5zcGxpdChpbmRpY2VzLCBlbmRzKQogICAgaWYgYW55KGxlbihncm91cCkgPT0gMCBmb3IgZ3JvdXAgaW4gZ3JvdXBzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEYXRhc2V0IGlzIHRvbyBzbWFsbCBmb3IgcmVxdWVzdGVkIHNwbGl0IHJhdGlvcyIpCiAgICBhcnJheXM6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgZm9yIGdyb3VwIGluIGdyb3VwczoKICAgICAgICBhcnJheXMuZXh0ZW5kKFt4W2dyb3VwXSwgeVtncm91cF1dKQogICAgcmV0dXJuIFRhYnVsYXJTcGxpdHMoKmFycmF5cykKCgpkZWYgZGlzY292ZXJfc3BsaXRzKGFycmF5czogTWFwcGluZ1tzdHIsIG5wLm5kYXJyYXldKSAtPiBUYWJ1bGFyU3BsaXRzOgogICAgIiIiVmFsaWRhdGUgY29udmVudGlvbmFsIGB0cmFpbl94YCB0aHJvdWdoIGB0ZXN0X3lgIGtleXMgZnJvbSBhbiBOUFogY2FjaGUuIiIiCiAgICByZXF1aXJlZCA9ICgidHJhaW5feCIsICJ0cmFpbl95IiwgInZhbGlkYXRpb25feCIsICJ2YWxpZGF0aW9uX3kiLCAiY2FsaWJyYXRpb25feCIsICJjYWxpYnJhdGlvbl95IiwgInRlc3RfeCIsICJ0ZXN0X3kiKQogICAgbWlzc2luZyA9IFtrZXkgZm9yIGtleSBpbiByZXF1aXJlZCBpZiBrZXkgbm90IGluIGFycmF5c10KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJNaXNzaW5nIHNwbGl0IGFycmF5czoge21pc3Npbmd9IikKICAgIHJldHVybiBUYWJ1bGFyU3BsaXRzKCoobnAuYXNhcnJheShhcnJheXNba2V5XSkgZm9yIGtleSBpbiByZXF1aXJlZCkpCg==", "src/datasets/upstream.py": "IiIiRXhhY3QgUVJUIHVwc3RyZWFtIGRhdGEgYWNxdWlzaXRpb24gYW5kIHNwbGl0LWNhY2hlIGNvbnN0cnVjdGlvbi4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmZyb20gZGF0YXNldHMubWFuaWZlc3QgaW1wb3J0IHFydDU3X21hbmlmZXN0CmZyb20gZGF0YXNldHMuc3BsaXRzIGltcG9ydCBUYWJ1bGFyU3BsaXRzCmZyb20gdXRpbHMuaW8gaW1wb3J0IHNhdmVfYXJyYXlzCgoKZGVmIHFydF9zcGxpdCh4OiBucC5uZGFycmF5LCB5OiBucC5uZGFycmF5LCBzZWVkOiBpbnQgPSAwKSAtPiBUYWJ1bGFyU3BsaXRzOgogICAgIiIiUmVwcm9kdWNlIFFSVCdzIGBbLjY1LDAsLjEwLC4xNSwuMTBdYCByYW5kb20tc3BsaXQgYWxsb2NhdGlvbiBleGFjdGx5LiIiIgogICAgaWYgeC5zaGFwZVswXSAhPSB5LnNoYXBlWzBdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInggYW5kIHkgbXVzdCBoYXZlIG1hdGNoaW5nIGZpcnN0IGRpbWVuc2lvbiIpCiAgICBzaXplID0geC5zaGFwZVswXQogICAgbGVuZ3RocyA9IChucC5hcnJheShbLjY1LCAwLiwgLjEwLCAuMTUsIC4xMF0pICogc2l6ZSkuYXN0eXBlKGludCkKICAgIG92ZXJmbG93ID0gbWF4KDAsIGludChsZW5ndGhzWzNdIC0gMjA0OCkpCiAgICBsZW5ndGhzWzNdIC09IG92ZXJmbG93CiAgICBtYXNrID0gKGxlbmd0aHMgIT0gMCkgJiAobnAuYXJhbmdlKDUpICE9IDMpCiAgICBsZW5ndGhzW21hc2tdICs9IGludChvdmVyZmxvdyAvIGludChtYXNrLnN1bSgpKSkKICAgIGxlbmd0aHNbLTFdID0gc2l6ZSAtIGxlbmd0aHNbOi0xXS5zdW0oKQogICAgaW5kaWNlcyA9IHRvcmNoLnJhbmRwZXJtKHNpemUsIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzZWVkKSkubnVtcHkoKQogICAgdHJhaW4sIGludGVyLCB2YWxpZGF0aW9uLCBjYWxpYnJhdGlvbiwgdGVzdCA9IG5wLnNwbGl0KGluZGljZXMsIG5wLmN1bXN1bShsZW5ndGhzKVs6LTFdKQogICAgaWYgaW50ZXIuc2l6ZToKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiUVJUIGJhc2VsaW5lIGNvbmZpZ3VyYXRpb24gZXhwZWN0cyBhbiBlbXB0eSBpbnRlcmxlYXZpbmcgc3BsaXQiKQogICAgcmV0dXJuIFRhYnVsYXJTcGxpdHMoeFt0cmFpbl0sIHlbdHJhaW5dLCB4W3ZhbGlkYXRpb25dLCB5W3ZhbGlkYXRpb25dLCB4W2NhbGlicmF0aW9uXSwgeVtjYWxpYnJhdGlvbl0sIHhbdGVzdF0sIHlbdGVzdF0pCgoKZGVmIGRvd25sb2FkX2FuZF9jYWNoZV9xcnQ1Nyh1cHN0cmVhbV9yb290OiBzdHIgfCBQYXRoLCBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgsIGNhY2hlX3Jvb3Q6IHN0ciB8IFBhdGgsIHNlZWQ6IGludCA9IDApIC0+IGxpc3RbUGF0aF06CiAgICAiIiJVc2UgdXBzdHJlYW0gZG93bmxvYWQvbG9hZCBmdW5jdGlvbnMgYW5kIGNhY2hlIGV4YWN0IFFSVC1jb21wYXRpYmxlIHNwbGl0cy4KCiAgICBUaGUgdXBzdHJlYW0gcmVwb3NpdG9yeSBpcyBwbGFjZWQgdGVtcG9yYXJpbHkgb24gYHN5cy5wYXRoYDsgaXRzIG93bgogICAgZG93bmxvYWRlciByZW1haW5zIHRoZSBhdXRob3JpdGF0aXZlIGRhdGFzZXQgc291cmNlIGFuZCBsaWNlbnNpbmcgcGF0aC4KICAgICIiIgogICAgcm9vdCwgZGF0YSwgY2FjaGUgPSBQYXRoKHVwc3RyZWFtX3Jvb3QpLCBQYXRoKGRhdGFfcm9vdCksIFBhdGgoY2FjaGVfcm9vdCkKICAgIGlmIG5vdCAocm9vdCAvICJ1cSIpLmlzX2RpcigpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWlzc2luZyBRUlQgdXBzdHJlYW0gc291cmNlIGF0IHtyb290fSIpCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKHJvb3QpKQogICAgdHJ5OgogICAgICAgIGZyb20gdXEuZGF0YW1vZHVsZXMub3Blbm1sLmRvd25sb2FkX29wZW5tbCBpbXBvcnQgZG93bmxvYWRfb3Blbm1sX3N1aXRlLCBsb2FkX2RhdGFzZXQgYXMgbG9hZF9vcGVubWwKICAgICAgICBmcm9tIHVxLmRhdGFtb2R1bGVzLnVjaS5kb3dubG9hZF91Y2kgaW1wb3J0IGRvd25sb2FkX2FsbF91Y2ksIGxvYWRfZGF0YXNldCBhcyBsb2FkX3VjaQoKICAgICAgICBkb3dubG9hZF9hbGxfdWNpKGRhdGEpCiAgICAgICAgZm9yIHN1aXRlIGluICgyOTcsIDI5OSwgMjY5KToKICAgICAgICAgICAgZG93bmxvYWRfb3Blbm1sX3N1aXRlKHN1aXRlLCBkYXRhKQogICAgICAgIHNhdmVkOiBsaXN0W1BhdGhdID0gW10KICAgICAgICBmb3Igc3BlYyBpbiBxcnQ1N19tYW5pZmVzdCgpOgogICAgICAgICAgICBsb2FkZXIgPSBsb2FkX3VjaSBpZiBzcGVjLmdyb3VwID09ICJ1Y2kiIGVsc2UgbG9hZF9vcGVubWwKICAgICAgICAgICAgc291cmNlID0gZGF0YSAvICgidWNpIiBpZiBzcGVjLmdyb3VwID09ICJ1Y2kiIGVsc2UgIm9wZW5tbCIpIC8gKCIiIGlmIHNwZWMuZ3JvdXAgPT0gInVjaSIgZWxzZSBzcGVjLmdyb3VwLnNwbGl0KCJfIilbMV0pIC8gc3BlYy5uYW1lCiAgICAgICAgICAgIHgsIHkgPSBsb2FkZXIoc291cmNlKQogICAgICAgICAgICBzcGxpdHMgPSBxcnRfc3BsaXQobnAuYXNhcnJheSh4KSwgbnAuYXNhcnJheSh5KSwgc2VlZCkKICAgICAgICAgICAgc2F2ZWQuYXBwZW5kKHNhdmVfYXJyYXlzKGNhY2hlIC8gZiJ7c3BlYy5uYW1lfS5ucHoiLCAqKnNwbGl0cy5fX2RpY3RfXykpCiAgICAgICAgcmV0dXJuIHNhdmVkCiAgICBmaW5hbGx5OgogICAgICAgIHN5cy5wYXRoLnJlbW92ZShzdHIocm9vdCkpCg==", "src/geometry/__init__.py": "IiIiUHVibGlzaGVkIE5SQyBtZXRyaWNzIGFuZCBleHBsaWNpdGx5IHByb3Bvc2VkIE5SQy1DYWwgZ2VvbWV0cnkgZGlzdGFuY2VzLiIiIgo=", "src/geometry/nrc.py": "IiIiRXhhY3QgcHVibGlzaGVkIE5SQyBtZXRyaWNzIHBsdXMgc2VwYXJhdGVseSBsYWJlbGxlZCBOUkMtQ2FsIGRpc3RhbmNlcy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBMaXRlcmFsCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5saW5hbGcgaW1wb3J0IHNxcnRtCmZyb20gc2NpcHkub3B0aW1pemUgaW1wb3J0IG1pbmltaXplX3NjYWxhcgoKTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgTlJDUmVzdWx0OgogICAgIiIiUHVibGlzaGVkIE5SQyBtZXRyaWNzIGFuZCB0aGUgcHJvcG9zZWQgY29uc2lzdGVudCBkaXN0YW5jZSBkZWNvbXBvc2l0aW9uLiIiIgoKICAgIG5yYzE6IGZsb2F0CiAgICBucmMyOiBmbG9hdAogICAgbnJjMzogZmxvYXQKICAgIGdhbW1hOiBmbG9hdCB8IE5vbmUKICAgIHJlc2lkdWFsX25yYzE6IG5wLm5kYXJyYXkKICAgIHJlc2lkdWFsX25yYzI6IG5wLm5kYXJyYXkKICAgIHNhbXBsZV9kaXN0YW5jZTogbnAubmRhcnJheQogICAgZGF0YXNldF9kaXN0YW5jZTogZmxvYXQKICAgIHdlaWdodHM6IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdCiAgICBub3JtYWxpemF0aW9uOiBzdHIKCgpkZWYgX3VuaXRfcm93cyhmZWF0dXJlczogbnAubmRhcnJheSwgY2VudGVyZWQ6IGJvb2wsIGVwc2lsb246IGZsb2F0KSAtPiBucC5uZGFycmF5OgogICAgIiIiTm9ybWFsaXplIGZlYXR1cmVzIGV4YWN0bHkgYXMgZWl0aGVyIGNpdGVkIE5SQyBwYXBlciBzcGVjaWZpZXMuIiIiCiAgICBtYXRyaXggPSBucC5hc2FycmF5KGZlYXR1cmVzLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgaWYgbWF0cml4Lm5kaW0gIT0gMiBvciBtYXRyaXguc2hhcGVbMF0gPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImZlYXR1cmVzIG11c3QgaGF2ZSBzaGFwZSBbZXhhbXBsZXMsIGZlYXR1cmVfZGltZW5zaW9uXSB3aXRoIGF0IGxlYXN0IHR3byBleGFtcGxlcyIpCiAgICBpZiBjZW50ZXJlZDoKICAgICAgICBtYXRyaXggPSBtYXRyaXggLSBtYXRyaXgubWVhbihheGlzPTAsIGtlZXBkaW1zPVRydWUpCiAgICBub3JtcyA9IG5wLmxpbmFsZy5ub3JtKG1hdHJpeCwgYXhpcz0xLCBrZWVwZGltcz1UcnVlKQogICAgaWYgbnAuYW55KG5vcm1zIDw9IGVwc2lsb24pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk5SQyBub3JtYWxpemF0aW9uIGlzIHVuZGVmaW5lZCBmb3IgYSB6ZXJvIGZlYXR1cmUgdmVjdG9yIikKICAgIHJldHVybiBtYXRyaXggLyBub3JtcwoKCmRlZiBfb3J0aG9ub3JtYWxfY29sdW1ucyhtYXRyaXg6IG5wLm5kYXJyYXksIHJhbms6IGludCwgZXBzaWxvbjogZmxvYXQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gYSBzdGFibGUgdGhpbiBvcnRob25vcm1hbCBiYXNpcyBmb3IgYSBjb2x1bW4gc3BhY2UuIiIiCiAgICBiYXNpcywgc2luZ3VsYXJfdmFsdWVzLCBfID0gbnAubGluYWxnLnN2ZChtYXRyaXgsIGZ1bGxfbWF0cmljZXM9RmFsc2UpCiAgICB2YWxpZCA9IGludChucC5zdW0oc2luZ3VsYXJfdmFsdWVzID4gZXBzaWxvbikpCiAgICBpZiB2YWxpZCA8IHJhbms6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlJlcXVpcmVkIHN1YnNwYWNlIHJhbmsge3Jhbmt9LCBvYnNlcnZlZCBudW1lcmljYWwgcmFuayB7dmFsaWR9IikKICAgIHJldHVybiBiYXNpc1s6LCA6cmFua10KCgpkZWYgX3Byb2plY3Rpb25fcmVzaWR1YWxzKHVuaXRfZmVhdHVyZXM6IG5wLm5kYXJyYXksIGJhc2lzOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV0dXJuIHNxdWFyZWQgcmVzaWR1YWxzIGFmdGVyIG9ydGhvZ29uYWwgcHJvamVjdGlvbiBvbnRvIGBiYXNpc2AuIiIiCiAgICBwcm9qZWN0ZWQgPSAodW5pdF9mZWF0dXJlcyBAIGJhc2lzKSBAIGJhc2lzLlQKICAgIHJldHVybiBucC5laW5zdW0oImlqLGlqLT5pIiwgdW5pdF9mZWF0dXJlcyAtIHByb2plY3RlZCwgdW5pdF9mZWF0dXJlcyAtIHByb2plY3RlZCkKCgpkZWYgdGFyZ2V0X2NvdmFyaWFuY2UodGFyZ2V0czogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICIiIkNvbXB1dGUgdGhlIHBhcGVyJ3MgYE1eLTFgIGVtcGlyaWNhbCB0YXJnZXQgY292YXJpYW5jZSBtYXRyaXguIiIiCiAgICB5ID0gbnAuYXNhcnJheSh0YXJnZXRzLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgaWYgeS5uZGltID09IDE6CiAgICAgICAgeSA9IHlbOiwgTm9uZV0KICAgIGlmIHkubmRpbSAhPSAyIG9yIHkuc2hhcGVbMF0gPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRhcmdldHMgbXVzdCBoYXZlIHNoYXBlIFtleGFtcGxlcywgdGFyZ2V0X2RpbWVuc2lvbl0iKQogICAgY2VudGVyZWQgPSB5IC0geS5tZWFuKGF4aXM9MCwga2VlcGRpbXM9VHJ1ZSkKICAgIGNvdmFyaWFuY2UgPSBjZW50ZXJlZC5UIEAgY2VudGVyZWQgLyB5LnNoYXBlWzBdCiAgICByZXR1cm4gKGNvdmFyaWFuY2UgKyBjb3ZhcmlhbmNlLlQpIC8gMi4wCgoKZGVmIF9wdWJsaXNoZWRfbnJjMyh3ZWlnaHQ6IG5wLm5kYXJyYXksIGNvdmFyaWFuY2U6IG5wLm5kYXJyYXksIGVwc2lsb246IGZsb2F0KSAtPiB0dXBsZVtmbG9hdCwgZmxvYXQgfCBOb25lXToKICAgICIiIkNvbXB1dGUgTmV1cklQUy0yMDI0IE5SQzMgYnkgYm91bmRlZCBtaW5pbWl6YXRpb24gb3ZlciBwdWJsaXNoZWQgZ2FtbWEuIiIiCiAgICBuID0gY292YXJpYW5jZS5zaGFwZVswXQogICAgaWYgbiA9PSAxOgogICAgICAgIHJldHVybiAwLjAsIE5vbmUgICMgVGhlIHNvdXJjZSBwYXBlciBleHBsaWNpdGx5IGNhbGxzIHVuaXZhcmlhdGUgTlJDMyB0cml2aWFsLgogICAgZWlndmFscyA9IG5wLmxpbmFsZy5laWd2YWxzaChjb3ZhcmlhbmNlKQogICAgbGFtYmRhX21pbiA9IGZsb2F0KGVpZ3ZhbHNbMF0pCiAgICBpZiBsYW1iZGFfbWluIDw9IGVwc2lsb246CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUHVibGlzaGVkIE5SQzMgcmVxdWlyZXMgYSBmdWxsLXJhbmsgcG9zaXRpdmUtZGVmaW5pdGUgdGFyZ2V0IGNvdmFyaWFuY2UiKQogICAgc2lnbWFfaGFsZiA9IG5wLmFzYXJyYXkoc3FydG0oY292YXJpYW5jZSkucmVhbCwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGdyYW0gPSB3ZWlnaHQgQCB3ZWlnaHQuVAogICAgZ3JhbV9ub3JtID0gbnAubGluYWxnLm5vcm0oZ3JhbSwgb3JkPSJmcm8iKQogICAgaWYgZ3JhbV9ub3JtIDw9IGVwc2lsb246CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUHVibGlzaGVkIG5vcm1hbGl6ZWQgTlJDMyBpcyB1bmRlZmluZWQgZm9yIHplcm8gZmluYWwtbGF5ZXIgR3JhbSBtYXRyaXgiKQoKICAgIGRlZiBvYmplY3RpdmUoZ2FtbWE6IGZsb2F0KSAtPiBmbG9hdDoKICAgICAgICB0YXJnZXQgPSBzaWdtYV9oYWxmIC0gbnAuc3FydChnYW1tYSkgKiBucC5leWUobikKICAgICAgICB0YXJnZXRfbm9ybSA9IG5wLmxpbmFsZy5ub3JtKHRhcmdldCwgb3JkPSJmcm8iKQogICAgICAgIGlmIHRhcmdldF9ub3JtIDw9IGVwc2lsb246CiAgICAgICAgICAgIHJldHVybiBmbG9hdCgiaW5mIikKICAgICAgICByZXR1cm4gZmxvYXQobnAubGluYWxnLm5vcm0oZ3JhbSAvIGdyYW1fbm9ybSAtIHRhcmdldCAvIHRhcmdldF9ub3JtLCBvcmQ9ImZybyIpICoqIDIpCgogICAgdXBwZXIgPSBucC5uZXh0YWZ0ZXIobGFtYmRhX21pbiwgMC4wKQogICAgcmVzdWx0ID0gbWluaW1pemVfc2NhbGFyKG9iamVjdGl2ZSwgYm91bmRzPShlcHNpbG9uLCB1cHBlciksIG1ldGhvZD0iYm91bmRlZCIpCiAgICBpZiBub3QgcmVzdWx0LnN1Y2Nlc3M6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTlJDMyBnYW1tYSBtaW5pbWl6YXRpb24gZmFpbGVkOiB7cmVzdWx0Lm1lc3NhZ2V9IikKICAgIHJldHVybiBmbG9hdChyZXN1bHQuZnVuKSwgZmxvYXQocmVzdWx0LngpCgoKZGVmIGNvbXB1dGVfbnJjKGZlYXR1cmVzOiBucC5uZGFycmF5LCB0YXJnZXRzOiBucC5uZGFycmF5LCBtZWFuX2hlYWRfd2VpZ2h0OiBucC5uZGFycmF5LCAqLCB3ZWlnaHRzOiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XSA9ICgxIC8gMywgMSAvIDMsIDEgLyAzKSwgbm9ybWFsaXphdGlvbjogTGl0ZXJhbFsibmV1cmlwc18yMDI0IiwgImludHJpbnNpY19kaW1lbnNpb25fMjAyNSJdID0gIm5ldXJpcHNfMjAyNCIsIGVwc2lsb246IGZsb2F0ID0gMWUtMTIpIC0+IE5SQ1Jlc3VsdDoKICAgICIiIkNvbXB1dGUgY2l0ZWQgTlJDMS0tMyBhbmQgcHJvcG9zZWQgTlJDLUNhbCBzYW1wbGUvZGF0YXNldCBkaXN0YW5jZXMuCgogICAgTlJDMS0tMyBmb2xsb3cgdGhlIGNpdGVkIGRlZmluaXRpb25zLiBgc2FtcGxlX2Rpc3RhbmNlYCBhbmQKICAgIGBkYXRhc2V0X2Rpc3RhbmNlYCBhcmUgTlJDLUNhbCBwcm9wb3NhbCBlcXVhdGlvbnMgZG9jdW1lbnRlZCBpbgogICAgYGRvY3MvbWV0aG9kb2xvZ3kubWRgLCBuZXZlciBwdWJsaXNoZWQgTlJDIG1ldHJpY3MuCiAgICAiIiIKICAgIGggPSBucC5hc2FycmF5KGZlYXR1cmVzLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgeSA9IG5wLmFzYXJyYXkodGFyZ2V0cywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIHcgPSBucC5hc2FycmF5KG1lYW5faGVhZF93ZWlnaHQsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBpZiB5Lm5kaW0gPT0gMToKICAgICAgICB5ID0geVs6LCBOb25lXQogICAgaWYgaC5zaGFwZVswXSAhPSB5LnNoYXBlWzBdIG9yIHcuc2hhcGUgIT0gKHkuc2hhcGVbMV0sIGguc2hhcGVbMV0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIGZlYXR1cmVzIFtNLGRdLCB0YXJnZXRzIFtNLG5dLCBhbmQgbWVhbl9oZWFkX3dlaWdodCBbbixkXSIpCiAgICBpZiBucC5hbnkobnAuYXNhcnJheSh3ZWlnaHRzKSA8IDApIG9yIG5vdCBucC5pc2Nsb3NlKHN1bSh3ZWlnaHRzKSwgMS4wKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJOUkMtQ2FsIHdlaWdodHMgbXVzdCBiZSBub25uZWdhdGl2ZSBhbmQgc3VtIHRvIG9uZSIpCiAgICBjZW50ZXJlZCA9IG5vcm1hbGl6YXRpb24gPT0gImludHJpbnNpY19kaW1lbnNpb25fMjAyNSIKICAgIHVuaXQgPSBfdW5pdF9yb3dzKGgsIGNlbnRlcmVkLCBlcHNpbG9uKQogICAgbiA9IHkuc2hhcGVbMV0KICAgICMgUENBIGNvbHVtbnMgYXJlIHJpZ2h0IHNpbmd1bGFyIHZlY3RvcnMgb2YgdGhlIGV4YW1wbGVzLWJ5LWZlYXR1cmVzIG1hdHJpeC4KICAgIF8sIF8sIHJpZ2h0X3ZlY3RvcnMgPSBucC5saW5hbGcuc3ZkKGggLSAoaC5tZWFuKDAsIGtlZXBkaW1zPVRydWUpIGlmIGNlbnRlcmVkIGVsc2UgMC4wKSwgZnVsbF9tYXRyaWNlcz1GYWxzZSkKICAgIGlmIHJpZ2h0X3ZlY3RvcnMuc2hhcGVbMF0gPCBuOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZlYXR1cmUgZGltZW5zaW9uIG11c3QgYmUgYXQgbGVhc3QgdGFyZ2V0IGRpbWVuc2lvbiIpCiAgICBwY2FfYmFzaXMgPSByaWdodF92ZWN0b3JzWzpuXS5UCiAgICB3ZWlnaHRfYmFzaXMgPSBfb3J0aG9ub3JtYWxfY29sdW1ucyh3LlQsIG4sIGVwc2lsb24pCiAgICByZXNpZHVhbDEgPSBfcHJvamVjdGlvbl9yZXNpZHVhbHModW5pdCwgcGNhX2Jhc2lzKQogICAgcmVzaWR1YWwyID0gX3Byb2plY3Rpb25fcmVzaWR1YWxzKHVuaXQsIHdlaWdodF9iYXNpcykKICAgIG5yYzEsIG5yYzIgPSBmbG9hdChyZXNpZHVhbDEubWVhbigpKSwgZmxvYXQocmVzaWR1YWwyLm1lYW4oKSkKICAgIG5yYzMsIGdhbW1hID0gX3B1Ymxpc2hlZF9ucmMzKHcsIHRhcmdldF9jb3ZhcmlhbmNlKHkpLCBlcHNpbG9uKQogICAgcHJvcG9zZWRfd2VpZ2h0cyA9IG5wLmFzYXJyYXkod2VpZ2h0cywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIG4gPT0gMSBhbmQgcHJvcG9zZWRfd2VpZ2h0c1syXSAhPSAwLjA6CiAgICAgICAgcHJvcG9zZWRfd2VpZ2h0c1syXSA9IDAuMAogICAgICAgIHByb3Bvc2VkX3dlaWdodHMgLz0gcHJvcG9zZWRfd2VpZ2h0cy5zdW0oKQogICAgICAgIExPR0dFUi5pbmZvKCJSZW5vcm1hbGl6ZWQgTlJDLUNhbCB3ZWlnaHRzIGJlY2F1c2UgcHVibGlzaGVkIHVuaXZhcmlhdGUgTlJDMyBpcyB0cml2aWFsIikKICAgIHNhbXBsZSA9IHByb3Bvc2VkX3dlaWdodHNbMF0gKiByZXNpZHVhbDEgKyBwcm9wb3NlZF93ZWlnaHRzWzFdICogcmVzaWR1YWwyICsgcHJvcG9zZWRfd2VpZ2h0c1syXSAqIG5yYzMKICAgIGRhdGFzZXQgPSBwcm9wb3NlZF93ZWlnaHRzWzBdICogbnJjMSArIHByb3Bvc2VkX3dlaWdodHNbMV0gKiBucmMyICsgcHJvcG9zZWRfd2VpZ2h0c1syXSAqIG5yYzMKICAgIGlmIG5vdCBucC5pc2Nsb3NlKHNhbXBsZS5tZWFuKCksIGRhdGFzZXQsIGF0b2w9MWUtMTApOgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJOUkMtQ2FsIHNhbXBsZS9kYXRhc2V0IGNvbnNpc3RlbmN5IGludmFyaWFudCBmYWlsZWQiKQogICAgcmV0dXJuIE5SQ1Jlc3VsdChucmMxLCBucmMyLCBucmMzLCBnYW1tYSwgcmVzaWR1YWwxLCByZXNpZHVhbDIsIHNhbXBsZSwgZmxvYXQoZGF0YXNldCksIHR1cGxlKHByb3Bvc2VkX3dlaWdodHMpLCBub3JtYWxpemF0aW9uKQo=", "src/metrics/__init__.py": "IiIiRGlzdHJpYnV0aW9uYWwgcmVncmVzc2lvbiBtZXRyaWNzLiIiIgo=", "src/metrics/evaluation.py": "IiIiRXZhbHVhdGlvbiBtZXRyaWNzIHVzZWQgY29uc2lzdGVudGx5IGFjcm9zcyBOUkMtQ2FsIGV4cGVyaW1lbnRzLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgbm9ybQoKZnJvbSBtb2RlbHMucHJlZGljdGlvbnMgaW1wb3J0IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb24KCgpkZWYgcHJvYmFiaWxpdHlfY2FsaWJyYXRpb25fZXJyb3IocHJlZGljdGlvbjogR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbiwgdGFyZ2V0czogbnAubmRhcnJheSwgbGV2ZWxzOiBpbnQgPSAxMDApIC0+IGZsb2F0OgogICAgIiIiQ29tcHV0ZSBRUlQtc3R5bGUgUENFOiBtZWFuIGFic29sdXRlIGVtcGlyaWNhbCBDREYgZXJyb3Igb3ZlciBsZXZlbHMuIiIiCiAgICBpZiBsZXZlbHMgPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxldmVscyBtdXN0IGJlIGF0IGxlYXN0IHR3byIpCiAgICBwaXQgPSBwcmVkaWN0aW9uLmNkZl8xZCh0YXJnZXRzKQogICAgYWxwaGEgPSBucC5saW5zcGFjZSgxIC8gbGV2ZWxzLCAxLjAsIGxldmVscykKICAgIHJldHVybiBmbG9hdChucC5tZWFuKG5wLmFicygocGl0WzosIE5vbmVdIDw9IGFscGhhKS5tZWFuKGF4aXM9MCkgLSBhbHBoYSkpKQoKCmRlZiBuZWdhdGl2ZV9sb2dfbGlrZWxpaG9vZChwcmVkaWN0aW9uOiBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uLCB0YXJnZXRzOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICIiIlJldHVybiBhdmVyYWdlIG5lZ2F0aXZlIG1peHR1cmUgbG9nIGxpa2VsaWhvb2QuIiIiCiAgICByZXR1cm4gZmxvYXQoLXByZWRpY3Rpb24ubG9ncGRmKHRhcmdldHMpLm1lYW4oKSkKCgpkZWYgcm1zZShwcmVkaWN0aW9uOiBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uLCB0YXJnZXRzOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICIiIlJldHVybiByb290IG1lYW4gc3F1YXJlZCBlcnJvciBvZiBwcmVkaWN0aXZlIG1lYW4uIiIiCiAgICByZXR1cm4gZmxvYXQobnAuc3FydChucC5tZWFuKChwcmVkaWN0aW9uLm1lYW4gLSBucC5hc2FycmF5KHRhcmdldHMpKSAqKiAyKSkpCgoKZGVmIG1hZShwcmVkaWN0aW9uOiBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uLCB0YXJnZXRzOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICIiIlJldHVybiBtZWFuIGFic29sdXRlIGVycm9yIG9mIHByZWRpY3RpdmUgbWVhbi4iIiIKICAgIHJldHVybiBmbG9hdChucC5tZWFuKG5wLmFicyhwcmVkaWN0aW9uLm1lYW4gLSBucC5hc2FycmF5KHRhcmdldHMpKSkpCgoKZGVmIGNvdmVyYWdlX2FuZF9zaGFycG5lc3MocHJlZGljdGlvbjogR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbiwgdGFyZ2V0czogbnAubmRhcnJheSwgbGV2ZWw6IGZsb2F0ID0gMC45KSAtPiB0dXBsZVtmbG9hdCwgZmxvYXRdOgogICAgIiIiUmV0dXJuIHVuaXZhcmlhdGUgY2VudHJhbC1pbnRlcnZhbCBjb3ZlcmFnZSBhbmQgbWVhbiBpbnRlcnZhbCB3aWR0aC4iIiIKICAgIGlmIHByZWRpY3Rpb24ubWVhbnMuc2hhcGVbMl0gIT0gMSBvciBub3QgKDAgPCBsZXZlbCA8IDEpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNvdmVyYWdlIGlzIGN1cnJlbnRseSBkZWZpbmVkIGZvciB1bml2YXJpYXRlIHRhcmdldHMgYW5kIDAgPCBsZXZlbCA8IDEiKQogICAgeSA9IG5wLmFzYXJyYXkodGFyZ2V0cykucmVzaGFwZSgtMSkKICAgIHogPSBub3JtLnBwZigoMS4wICsgbGV2ZWwpIC8gMi4wKQogICAgdmFyaWFuY2UgPSBwcmVkaWN0aW9uLnRvdGFsX2NvdmFyaWFuY2VbOiwgMCwgMF0KICAgIGhhbGZfd2lkdGggPSB6ICogbnAuc3FydCh2YXJpYW5jZSkKICAgIG1lYW4gPSBwcmVkaWN0aW9uLm1lYW5bOiwgMF0KICAgIHJldHVybiBmbG9hdChucC5tZWFuKG5wLmFicyh5IC0gbWVhbikgPD0gaGFsZl93aWR0aCkpLCBmbG9hdChucC5tZWFuKDIuMCAqIGhhbGZfd2lkdGgpKQoKCmRlZiBjcnBzX2dhdXNzaWFuX21vbWVudF9tYXRjaChwcmVkaWN0aW9uOiBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uLCB0YXJnZXRzOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICIiIkNvbXB1dGUgYSBkb2N1bWVudGVkIG1vbWVudC1tYXRjaGVkIEdhdXNzaWFuIENSUFMgZm9yIHVuaXZhcmlhdGUgbWl4dHVyZXMuIiIiCiAgICBpZiBwcmVkaWN0aW9uLm1lYW5zLnNoYXBlWzJdICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ1JQUyBpcyBpbXBsZW1lbnRlZCBmb3IgdW5pdmFyaWF0ZSBwcmVkaWN0aW9ucyIpCiAgICBtdSA9IHByZWRpY3Rpb24ubWVhbls6LCAwXQogICAgc2lnbWEgPSBucC5zcXJ0KHByZWRpY3Rpb24udG90YWxfY292YXJpYW5jZVs6LCAwLCAwXSkKICAgIHogPSAobnAuYXNhcnJheSh0YXJnZXRzKS5yZXNoYXBlKC0xKSAtIG11KSAvIHNpZ21hCiAgICBjcnBzID0gc2lnbWEgKiAoeiAqICgyICogbm9ybS5jZGYoeikgLSAxKSArIDIgKiBub3JtLnBkZih6KSAtIDEgLyBucC5zcXJ0KG5wLnBpKSkKICAgIHJldHVybiBmbG9hdChjcnBzLm1lYW4oKSkKCgpkZWYgZXZhbHVhdGUocHJlZGljdGlvbjogR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbiwgdGFyZ2V0czogbnAubmRhcnJheSkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICIiIkNvbXB1dGUgYWxsIGFwcGxpY2FibGUgY29yZSBtZXRyaWNzIGluIG9uZSBzdGFibGUgcmVzdWx0IGRpY3Rpb25hcnkuIiIiCiAgICB2YWx1ZXMgPSB7Im5sbCI6IG5lZ2F0aXZlX2xvZ19saWtlbGlob29kKHByZWRpY3Rpb24sIHRhcmdldHMpLCAicm1zZSI6IHJtc2UocHJlZGljdGlvbiwgdGFyZ2V0cyksICJtYWUiOiBtYWUocHJlZGljdGlvbiwgdGFyZ2V0cyl9CiAgICBpZiBwcmVkaWN0aW9uLm1lYW5zLnNoYXBlWzJdID09IDE6CiAgICAgICAgY292ZXJhZ2UsIHNoYXJwbmVzcyA9IGNvdmVyYWdlX2FuZF9zaGFycG5lc3MocHJlZGljdGlvbiwgdGFyZ2V0cykKICAgICAgICB2YWx1ZXMudXBkYXRlKHsicGNlIjogcHJvYmFiaWxpdHlfY2FsaWJyYXRpb25fZXJyb3IocHJlZGljdGlvbiwgdGFyZ2V0cyksICJjcnBzIjogY3Jwc19nYXVzc2lhbl9tb21lbnRfbWF0Y2gocHJlZGljdGlvbiwgdGFyZ2V0cyksICJjb3ZlcmFnZV85MCI6IGNvdmVyYWdlLCAic2hhcnBuZXNzXzkwIjogc2hhcnBuZXNzfSkKICAgIHJldHVybiB2YWx1ZXMK", "src/metrics/experiment.py": "IiIiQXJ0aWZhY3QtZHJpdmVuLCBmcm96ZW4tbW9kZWwgTlJDLUNhbCBleHBlcmltZW50IGV4ZWN1dGlvbi4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gY2FsaWJyYXRpb24ubnJjX2NhbCBpbXBvcnQgc2VsZWN0X3JpZGdlCmZyb20gZ2VvbWV0cnkubnJjIGltcG9ydCBjb21wdXRlX25yYwpmcm9tIG1ldHJpY3MuZXZhbHVhdGlvbiBpbXBvcnQgZXZhbHVhdGUKZnJvbSBtb2RlbHMucHJlZGljdGlvbnMgaW1wb3J0IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb24KZnJvbSB1dGlscy5pbyBpbXBvcnQgc2F2ZV9mcmFtZSwgc2F2ZV9qc29uCgoKZGVmIGxvYWRfcHJlZGljdGlvbl9jYWNoZShwYXRoOiBzdHIgfCBQYXRoKSAtPiB0dXBsZVtHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uLCBucC5uZGFycmF5XToKICAgICIiIkxvYWQgYSB2YWxpZGF0ZWQgcHJlZGljdGlvbiBjYWNoZSBwcm9kdWNlZCBieSB0aGUgZmVhdHVyZSBub3RlYm9vay4KCiAgICBSZXF1aXJlZCBrZXlzIGFyZSBgd2VpZ2h0c2AsIGBtZWFuc2AsIGBjb3ZhcmlhbmNlc2AsIGFuZCBgdGFyZ2V0c2A7IHRoaXMKICAgIGludGVudGlvbmFsbHkgcmVqZWN0cyBhbWJpZ3VvdXMgbW9kZWwtb3V0cHV0IGVuY29kaW5ncyBpbnN0ZWFkIG9mIGd1ZXNzaW5nCiAgICB3aGV0aGVyIGEgdGVuc29yIHJlcHJlc2VudHMgYSB2YXJpYW5jZSwgc3RhbmRhcmQgZGV2aWF0aW9uLCBvciBsb2cgc2NhbGUuCiAgICAiIiIKICAgIGNhY2hlID0gbnAubG9hZChQYXRoKHBhdGgpKQogICAgcmVxdWlyZWQgPSB7IndlaWdodHMiLCAibWVhbnMiLCAiY292YXJpYW5jZXMiLCAidGFyZ2V0cyJ9CiAgICBtaXNzaW5nID0gcmVxdWlyZWQuZGlmZmVyZW5jZShjYWNoZS5maWxlcykKICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJQcmVkaWN0aW9uIGNhY2hlIG1pc3NpbmcgcmVxdWlyZWQgYXJyYXlzOiB7c29ydGVkKG1pc3NpbmcpfSIpCiAgICByZXR1cm4gR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbihjYWNoZVsid2VpZ2h0cyJdLCBjYWNoZVsibWVhbnMiXSwgY2FjaGVbImNvdmFyaWFuY2VzIl0pLCBucC5hc2FycmF5KGNhY2hlWyJ0YXJnZXRzIl0pCgoKZGVmIHJ1bl9mcm96ZW5fbnJjX2NhbChkYXRhc2V0OiBzdHIsIGZhbWlseTogc3RyLCBjYWxpYnJhdGlvbl9mZWF0dXJlX2NhY2hlOiBzdHIgfCBQYXRoLCBjYWxpYnJhdGlvbl9wcmVkaWN0aW9uX2NhY2hlOiBzdHIgfCBQYXRoLCB0ZXN0X2ZlYXR1cmVfY2FjaGU6IHN0ciB8IFBhdGgsIHRlc3RfcHJlZGljdGlvbl9jYWNoZTogc3RyIHwgUGF0aCwgbWVhbl9oZWFkX3dlaWdodDogbnAubmRhcnJheSwgb3V0cHV0X3BhdGg6IHN0ciB8IFBhdGgpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkV4ZWN1dGUgQkFTRSBhbmQgTlJDLUNhbCBmcm9tIGltbXV0YWJsZSBmZWF0dXJlL3ByZWRpY3Rpb24gYXJ0aWZhY3RzLgoKICAgIFRoaXMgcnVubmVyIG5ldmVyIHRyYWlucyBvciBjaGFuZ2VzIGEgY2hlY2twb2ludC4gSXQgaXMgbW9kZWwtZmFtaWx5CiAgICBhZ25vc3RpYyBiZWNhdXNlIGFsbCBHYXVzc2lhbiBhbmQgbWl4dHVyZSBoZWFkcyBhcmUgbm9ybWFsaXplZCBpbnRvIHRoZQogICAgYEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb25gIGFydGlmYWN0IGNvbnRyYWN0LgogICAgIiIiCiAgICBjYWxpYnJhdGlvbl9mZWF0dXJlcyA9IG5wLmxvYWQoUGF0aChjYWxpYnJhdGlvbl9mZWF0dXJlX2NhY2hlKSkKICAgIHRlc3RfZmVhdHVyZXMgPSBucC5sb2FkKFBhdGgodGVzdF9mZWF0dXJlX2NhY2hlKSkKICAgIGNhbGlicmF0aW9uX3ByZWRpY3Rpb24sIGNhbGlicmF0aW9uX3RhcmdldHMgPSBsb2FkX3ByZWRpY3Rpb25fY2FjaGUoY2FsaWJyYXRpb25fcHJlZGljdGlvbl9jYWNoZSkKICAgIHRlc3RfcHJlZGljdGlvbiwgdGVzdF90YXJnZXRzID0gbG9hZF9wcmVkaWN0aW9uX2NhY2hlKHRlc3RfcHJlZGljdGlvbl9jYWNoZSkKICAgIGNhbGlicmF0aW9uX25yYyA9IGNvbXB1dGVfbnJjKGNhbGlicmF0aW9uX2ZlYXR1cmVzWyJmZWF0dXJlcyJdLCBjYWxpYnJhdGlvbl9mZWF0dXJlc1sidGFyZ2V0cyJdLCBtZWFuX2hlYWRfd2VpZ2h0KQogICAgdGVzdF9ucmMgPSBjb21wdXRlX25yYyh0ZXN0X2ZlYXR1cmVzWyJmZWF0dXJlcyJdLCB0ZXN0X2ZlYXR1cmVzWyJ0YXJnZXRzIl0sIG1lYW5faGVhZF93ZWlnaHQpCiAgICBjYWxpYnJhdG9yID0gc2VsZWN0X3JpZGdlKGNhbGlicmF0aW9uX3ByZWRpY3Rpb24sIGNhbGlicmF0aW9uX3RhcmdldHMsIGNhbGlicmF0aW9uX25yYy5zYW1wbGVfZGlzdGFuY2UpCiAgICBjYWxpYnJhdGVkX3Rlc3QgPSBjYWxpYnJhdG9yLnRyYW5zZm9ybSh0ZXN0X3ByZWRpY3Rpb24sIHRlc3RfbnJjLnNhbXBsZV9kaXN0YW5jZSkKICAgIHJvd3M6IGxpc3RbZGljdFtzdHIsIG9iamVjdF1dID0gW10KICAgIGZvciBtZXRob2QsIHByZWRpY3Rpb24gaW4gKCgiQkFTRSIsIHRlc3RfcHJlZGljdGlvbiksICgiTlJDLUNhbCIsIGNhbGlicmF0ZWRfdGVzdCkpOgogICAgICAgIHJvdzogZGljdFtzdHIsIG9iamVjdF0gPSB7ImRhdGFzZXQiOiBkYXRhc2V0LCAiZmFtaWx5IjogZmFtaWx5LCAibWV0aG9kIjogbWV0aG9kLCAibnJjX2Rpc3RhbmNlIjogdGVzdF9ucmMuZGF0YXNldF9kaXN0YW5jZSwgIm5yYzEiOiB0ZXN0X25yYy5ucmMxLCAibnJjMiI6IHRlc3RfbnJjLm5yYzIsICJucmMzIjogdGVzdF9ucmMubnJjMywgIm5yY19nYW1tYSI6IHRlc3RfbnJjLmdhbW1hfQogICAgICAgIHJvdy51cGRhdGUoZXZhbHVhdGUocHJlZGljdGlvbiwgdGVzdF90YXJnZXRzKSk7IHJvd3MuYXBwZW5kKHJvdykKICAgIGZyYW1lID0gcGQuRGF0YUZyYW1lKHJvd3MpCiAgICBzYXZlX2ZyYW1lKGZyYW1lLCBvdXRwdXRfcGF0aCkKICAgIHNhdmVfanNvbih7ImRhdGFzZXQiOiBkYXRhc2V0LCAiZmFtaWx5IjogZmFtaWx5LCAibnJjX2NhbGlicmF0b3IiOiBhc2RpY3QoY2FsaWJyYXRvciksICJwdWJsaXNoZWRfbnJjX25vcm1hbGl6YXRpb24iOiB0ZXN0X25yYy5ub3JtYWxpemF0aW9uLCAibnJjX2Rpc3RhbmNlX2lzX3Byb3Bvc2VkIjogVHJ1ZX0sIFBhdGgob3V0cHV0X3BhdGgpLndpdGhfc3VmZml4KCIucHJvdmVuYW5jZS5qc29uIikpCiAgICByZXR1cm4gZnJhbWUK", "src/metrics/statistics.py": "IiIiUGFpcmVkIHN0YXRpc3RpY2FsIHRlc3RzLCBtdWx0aXBsZS10ZXN0aW5nIGNvcnJlY3Rpb24sIGFuZCByYW5rIHN1bW1hcmllcy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgZnJpZWRtYW5jaGlzcXVhcmUsIGtlbmRhbGx0YXUsIHBlYXJzb25yLCBzcGVhcm1hbnIsIHdpbGNveG9uCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ29ycmVsYXRpb25SZXN1bHQ6CiAgICAiIiJDb3JyZWxhdGlvbiBlc3RpbWF0ZSwgcC12YWx1ZSwgYW5kIG5vbnBhcmFtZXRyaWMgYm9vdHN0cmFwIGludGVydmFsLiIiIgoKICAgIG5hbWU6IHN0cgogICAgY29lZmZpY2llbnQ6IGZsb2F0CiAgICBwdmFsdWU6IGZsb2F0CiAgICBjaV9sb3c6IGZsb2F0CiAgICBjaV9oaWdoOiBmbG9hdAogICAgcGVybXV0YXRpb25fcHZhbHVlOiBmbG9hdAoKCmRlZiBjb3JyZWxhdGlvbnMoeDogbnAubmRhcnJheSwgeTogbnAubmRhcnJheSwgYm9vdHN0cmFwX3NhbXBsZXM6IGludCA9IDJfMDAwLCBwZXJtdXRhdGlvbnM6IGludCA9IDVfMDAwLCBzZWVkOiBpbnQgPSAwKSAtPiB0dXBsZVtDb3JyZWxhdGlvblJlc3VsdCwgLi4uXToKICAgICIiIkNvbXB1dGUgUGVhcnNvbi9TcGVhcm1hbi9LZW5kYWxsIHdpdGggYm9vdHN0cmFwIENJcyBhbmQgcGVybXV0YXRpb24gdGVzdHMuIiIiCiAgICBhLCBiID0gbnAuYXNhcnJheSh4LCBmbG9hdCkucmVzaGFwZSgtMSksIG5wLmFzYXJyYXkoeSwgZmxvYXQpLnJlc2hhcGUoLTEpCiAgICBpZiBhLnNpemUgIT0gYi5zaXplIG9yIGEuc2l6ZSA8IDQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ29ycmVsYXRpb24gcmVxdWlyZXMgZXF1YWwgYXJyYXlzIHdpdGggYXQgbGVhc3QgZm91ciBvYnNlcnZhdGlvbnMiKQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBmdW5jdGlvbnMgPSAoKCJwZWFyc29uIiwgbGFtYmRhIHUsIHY6IHBlYXJzb25yKHUsIHYpLnN0YXRpc3RpYywgcGVhcnNvbnIpLCAoInNwZWFybWFuIiwgbGFtYmRhIHUsIHY6IHNwZWFybWFucih1LCB2KS5zdGF0aXN0aWMsIHNwZWFybWFuciksICgia2VuZGFsbCIsIGxhbWJkYSB1LCB2OiBrZW5kYWxsdGF1KHUsIHYpLnN0YXRpc3RpYywga2VuZGFsbHRhdSkpCiAgICByZXN1bHQ6IGxpc3RbQ29ycmVsYXRpb25SZXN1bHRdID0gW10KICAgIGZvciBuYW1lLCBjb2VmZmljaWVudF9mdW5jdGlvbiwgdGVzdF9mdW5jdGlvbiBpbiBmdW5jdGlvbnM6CiAgICAgICAgb2JzZXJ2ZWQsIHB2YWx1ZSA9IHRlc3RfZnVuY3Rpb24oYSwgYikKICAgICAgICBkcmF3cyA9IG5wLmFycmF5KFtjb2VmZmljaWVudF9mdW5jdGlvbihhW2luZGV4XSwgYltpbmRleF0pIGZvciBpbmRleCBpbiBybmcuaW50ZWdlcnMoMCwgYS5zaXplLCAoYm9vdHN0cmFwX3NhbXBsZXMsIGEuc2l6ZSkpXSkKICAgICAgICBudWxsID0gbnAuYXJyYXkoW2NvZWZmaWNpZW50X2Z1bmN0aW9uKGEsIHJuZy5wZXJtdXRhdGlvbihiKSkgZm9yIF8gaW4gcmFuZ2UocGVybXV0YXRpb25zKV0pCiAgICAgICAgcGVybXV0YXRpb25fcCA9ICgxLjAgKyBucC5zdW0obnAuYWJzKG51bGwpID49IGFicyhvYnNlcnZlZCkpKSAvIChwZXJtdXRhdGlvbnMgKyAxLjApCiAgICAgICAgcmVzdWx0LmFwcGVuZChDb3JyZWxhdGlvblJlc3VsdChuYW1lLCBmbG9hdChvYnNlcnZlZCksIGZsb2F0KHB2YWx1ZSksIGZsb2F0KG5wLm5hbnF1YW50aWxlKGRyYXdzLCAuMDI1KSksIGZsb2F0KG5wLm5hbnF1YW50aWxlKGRyYXdzLCAuOTc1KSksIGZsb2F0KHBlcm11dGF0aW9uX3ApKSkKICAgIHJldHVybiB0dXBsZShyZXN1bHQpCgoKZGVmIGhvbG1fYWRqdXN0KHB2YWx1ZXM6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gSG9sbS1Cb25mZXJyb25pIGFkanVzdGVkIHAtdmFsdWVzIGluIG9yaWdpbmFsIG9yZGVyLiIiIgogICAgcCA9IG5wLmFzYXJyYXkocHZhbHVlcywgZmxvYXQpCiAgICBvcmRlciA9IG5wLmFyZ3NvcnQocCkKICAgIGFkanVzdGVkID0gbnAuZW1wdHlfbGlrZShwKQogICAgcnVubmluZyA9IDAuMAogICAgZm9yIHJhbmssIGluZGV4IGluIGVudW1lcmF0ZShvcmRlcik6CiAgICAgICAgcnVubmluZyA9IG1heChydW5uaW5nLCAocC5zaXplIC0gcmFuaykgKiBwW2luZGV4XSkKICAgICAgICBhZGp1c3RlZFtpbmRleF0gPSBtaW4ocnVubmluZywgMS4wKQogICAgcmV0dXJuIGFkanVzdGVkCgoKZGVmIHBhaXJlZF9zdGF0aXN0aWNzKHNjb3JlczogbnAubmRhcnJheSkgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICAiIiJSdW4gRnJpZWRtYW4gYW5kIHBhaXJ3aXNlIFdpbGNveG9uIHRlc3RzIGZvciBgW2RhdGFzZXRzLCBtZXRob2RzXWAgc2NvcmVzLiIiIgogICAgbWF0cml4ID0gbnAuYXNhcnJheShzY29yZXMsIGZsb2F0KQogICAgaWYgbWF0cml4Lm5kaW0gIT0gMiBvciBtYXRyaXguc2hhcGVbMV0gPCAzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk5lZWQgc2NvcmVzIGZvciBhdCBsZWFzdCB0aHJlZSBtZXRob2RzIikKICAgIGZyaWVkbWFuID0gZnJpZWRtYW5jaGlzcXVhcmUoKihtYXRyaXhbOiwgY29sdW1uXSBmb3IgY29sdW1uIGluIHJhbmdlKG1hdHJpeC5zaGFwZVsxXSkpKQogICAgcGFpcnM6IGxpc3RbdHVwbGVbaW50LCBpbnQsIGZsb2F0XV0gPSBbXQogICAgZm9yIGxlZnQgaW4gcmFuZ2UobWF0cml4LnNoYXBlWzFdKToKICAgICAgICBmb3IgcmlnaHQgaW4gcmFuZ2UobGVmdCArIDEsIG1hdHJpeC5zaGFwZVsxXSk6CiAgICAgICAgICAgIHBhaXJzLmFwcGVuZCgobGVmdCwgcmlnaHQsIGZsb2F0KHdpbGNveG9uKG1hdHJpeFs6LCBsZWZ0XSwgbWF0cml4WzosIHJpZ2h0XSwgemVyb19tZXRob2Q9IndpbGNveCIpLnB2YWx1ZSkpKQogICAgcmV0dXJuIHsiZnJpZWRtYW5fc3RhdGlzdGljIjogZmxvYXQoZnJpZWRtYW4uc3RhdGlzdGljKSwgImZyaWVkbWFuX3B2YWx1ZSI6IGZsb2F0KGZyaWVkbWFuLnB2YWx1ZSksICJwYWlycyI6IHBhaXJzLCAiaG9sbV9wdmFsdWVzIjogaG9sbV9hZGp1c3QobnAuYXJyYXkoW3BhaXJbMl0gZm9yIHBhaXIgaW4gcGFpcnNdKSkudG9saXN0KCksICJhdmVyYWdlX3JhbmtzIjogbnAubWVhbihucC5hcmdzb3J0KG5wLmFyZ3NvcnQobWF0cml4LCBheGlzPTEpLCBheGlzPTEpICsgMSwgYXhpcz0wKS50b2xpc3QoKX0K", "src/models/__init__.py": "IiIiRnJvemVuIHByZWRpY3Rpb24gY29udGFpbmVycyBhbmQgZmVhdHVyZSBleHRyYWN0aW9uIGFkYXB0ZXJzLiIiIgo=", "src/models/adapters.py": "IiIiQWRhcHRlcnMgZnJvbSBjb21tb24gZnJvemVuIEdhdXNzaWFuIGhlYWQgdGVuc29ycyB0byB2YWxpZGF0ZWQgcHJlZGljdGlvbnMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBtb2RlbHMucHJlZGljdGlvbnMgaW1wb3J0IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb24sIGdhdXNzaWFuX3ByZWRpY3Rpb24KCgpkZWYgcHJlZGljdGlvbl9mcm9tX2hlYWRzKG1lYW5zOiBucC5uZGFycmF5LCBsb2dfdmFyaWFuY2VzOiBucC5uZGFycmF5LCBsb2dpdHM6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSwgbWluaW11bV92YXJpYW5jZTogZmxvYXQgPSAxZS04KSAtPiBHYXVzc2lhbk1peHR1cmVQcmVkaWN0aW9uOgogICAgIiIiQnVpbGQgR2F1c3NpYW4sIGdlbmVyaWMgbWl4dHVyZSwgTWl4dHVyZS0zLCBvciBNaXh0dXJlLTEwIHByZWRpY3Rpb25zLgoKICAgIGBtZWFuc2AgYW5kIGBsb2dfdmFyaWFuY2VzYCBhY2NlcHQgYFtOLERdYCBmb3IgYSBHYXVzc2lhbiBvciBgW04sSyxEXWAKICAgIGZvciBhbnkgbWl4dHVyZSBjb3VudCBgS2AsIGluY2x1ZGluZyAzIGFuZCAxMC4gT3B0aW9uYWwgbG9naXRzIGhhdmUKICAgIHNoYXBlIGBbTixLXWAgYW5kIGFyZSBub3JtYWxpemVkIHdpdGggYSBzdGFibGUgc29mdG1heC4KICAgICIiIgogICAgbWVhbiA9IG5wLmFzYXJyYXkobWVhbnMsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICB2YXJpYW5jZSA9IG5wLm1heGltdW0obnAuZXhwKG5wLmFzYXJyYXkobG9nX3ZhcmlhbmNlcywgZHR5cGU9bnAuZmxvYXQ2NCkpLCBtaW5pbXVtX3ZhcmlhbmNlKQogICAgaWYgbWVhbi5uZGltID09IDI6CiAgICAgICAgcmV0dXJuIGdhdXNzaWFuX3ByZWRpY3Rpb24obWVhbiwgdmFyaWFuY2UsIG1pbmltdW1fdmFyaWFuY2UpCiAgICBpZiBtZWFuLm5kaW0gIT0gMyBvciBtZWFuLnNoYXBlICE9IHZhcmlhbmNlLnNoYXBlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIG1hdGNoaW5nIFtOLERdIG9yIFtOLEssRF0gbWVhbi9sb2ctdmFyaWFuY2UgdGVuc29ycyIpCiAgICBuX2V4YW1wbGVzLCBjb21wb25lbnRzLCBkaW1lbnNpb24gPSBtZWFuLnNoYXBlCiAgICBpZiBsb2dpdHMgaXMgTm9uZToKICAgICAgICB3ZWlnaHRzID0gbnAuZnVsbCgobl9leGFtcGxlcywgY29tcG9uZW50cyksIDEuMCAvIGNvbXBvbmVudHMpCiAgICBlbHNlOgogICAgICAgIHNjb3JlID0gbnAuYXNhcnJheShsb2dpdHMsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICAgICAgaWYgc2NvcmUuc2hhcGUgIT0gKG5fZXhhbXBsZXMsIGNvbXBvbmVudHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsb2dpdHMgbXVzdCBoYXZlIHNoYXBlIFtOLEtdIikKICAgICAgICBzY29yZSAtPSBzY29yZS5tYXgoYXhpcz0xLCBrZWVwZGltcz1UcnVlKQogICAgICAgIHdlaWdodHMgPSBucC5leHAoc2NvcmUpOyB3ZWlnaHRzIC89IHdlaWdodHMuc3VtKGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKICAgIGNvdmFyaWFuY2UgPSBucC5leWUoZGltZW5zaW9uKVtOb25lLCBOb25lXSAqIHZhcmlhbmNlWzosIDosIDosIE5vbmVdCiAgICByZXR1cm4gR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbih3ZWlnaHRzLCBtZWFuLCBjb3ZhcmlhbmNlKQo=", "src/models/features.py": "IiIiQXJjaGl0ZWN0dXJlLWFnbm9zdGljIGZyb3plbiBQeVRvcmNoIGZlYXR1cmUgZXh0cmFjdGlvbi4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgSXRlcmFibGUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBGZWF0dXJlQmF0Y2g6CiAgICAiIiJDYWNoZWQgZnJvemVuLW1vZGVsIGZlYXR1cmVzLCByYXcgbW9kZWwgb3V0cHV0LCBhbmQgdGFyZ2V0cy4iIiIKCiAgICBmZWF0dXJlczogbnAubmRhcnJheQogICAgb3V0cHV0czogbnAubmRhcnJheQogICAgdGFyZ2V0czogbnAubmRhcnJheQoKCmRlZiBleHRyYWN0X2ZlYXR1cmVzKG1vZGVsOiBubi5Nb2R1bGUsIGxheWVyOiBubi5Nb2R1bGUsIGxvYWRlcjogSXRlcmFibGVbdHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdXSwgZGV2aWNlOiB0b3JjaC5kZXZpY2UgfCBzdHIsIG91dHB1dF90cmFuc2Zvcm06IENhbGxhYmxlW1tvYmplY3RdLCB0b3JjaC5UZW5zb3JdIHwgTm9uZSA9IE5vbmUpIC0+IEZlYXR1cmVCYXRjaDoKICAgICIiIkZvcndhcmQgYSBsb2FkZXIgb25jZSB3aGlsZSBjYXB0dXJpbmcgYSBuYW1lZCBwZW51bHRpbWF0ZS1sYXllciBhY3RpdmF0aW9uLiIiIgogICAgY2FwdHVyZWQ6IGxpc3RbdG9yY2guVGVuc29yXSA9IFtdCgogICAgZGVmIGhvb2soXzogbm4uTW9kdWxlLCBfXzogdHVwbGVbdG9yY2guVGVuc29yLCAuLi5dLCBvdXRwdXQ6IHRvcmNoLlRlbnNvcikgLT4gTm9uZToKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShvdXRwdXQsIHRvcmNoLlRlbnNvcik6CiAgICAgICAgICAgIHJhaXNlIFR5cGVFcnJvcigiVGhlIHNlbGVjdGVkIGZlYXR1cmUgbGF5ZXIgbXVzdCBvdXRwdXQgYSB0ZW5zb3IiKQogICAgICAgIGNhcHR1cmVkLmFwcGVuZChvdXRwdXQuZGV0YWNoKCkuY3B1KCkpCgogICAgaGFuZGxlID0gbGF5ZXIucmVnaXN0ZXJfZm9yd2FyZF9ob29rKGhvb2spCiAgICBvdXRwdXRzOiBsaXN0W3RvcmNoLlRlbnNvcl0gPSBbXQogICAgdGFyZ2V0czogbGlzdFt0b3JjaC5UZW5zb3JdID0gW10KICAgIG1vZGVsLmV2YWwoKQogICAgdHJ5OgogICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICAgICAgZm9yIHgsIHkgaW4gbG9hZGVyOgogICAgICAgICAgICAgICAgcmF3ID0gbW9kZWwoeC50byhkZXZpY2UpKQogICAgICAgICAgICAgICAgdmFsdWUgPSBvdXRwdXRfdHJhbnNmb3JtKHJhdykgaWYgb3V0cHV0X3RyYW5zZm9ybSBlbHNlIHJhdwogICAgICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIHRvcmNoLlRlbnNvcik6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVHlwZUVycm9yKCJvdXRwdXRfdHJhbnNmb3JtIG11c3QgcmV0dXJuIGEgdGVuc29yIikKICAgICAgICAgICAgICAgIG91dHB1dHMuYXBwZW5kKHZhbHVlLmRldGFjaCgpLmNwdSgpKQogICAgICAgICAgICAgICAgdGFyZ2V0cy5hcHBlbmQoeS5kZXRhY2goKS5jcHUoKSkKICAgIGZpbmFsbHk6CiAgICAgICAgaGFuZGxlLnJlbW92ZSgpCiAgICBpZiBub3QgY2FwdHVyZWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJGZWF0dXJlIGV4dHJhY3Rpb24gcmVjZWl2ZWQgbm8gYmF0Y2hlcyIpCiAgICByZXR1cm4gRmVhdHVyZUJhdGNoKHRvcmNoLmNhdChjYXB0dXJlZCkubnVtcHkoKSwgdG9yY2guY2F0KG91dHB1dHMpLm51bXB5KCksIHRvcmNoLmNhdCh0YXJnZXRzKS5udW1weSgpKQo=", "src/models/predictions.py": "IiIiVmFsaWRhdGVkIEdhdXNzaWFuIGFuZCBHYXVzc2lhbi1taXh0dXJlIHByZWRpY3RpdmUgZGlzdHJpYnV0aW9ucy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5zcGVjaWFsIGltcG9ydCBsb2dzdW1leHAKZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgbXVsdGl2YXJpYXRlX25vcm1hbCwgbm9ybQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb246CiAgICAiIiJCYXRjaCBvZiBkaWFnb25hbC9mdWxsIEdhdXNzaWFuIG1peHR1cmVzIHdpdGggc2hhcGUgYFtOLEssRF1gIHBhcmFtZXRlcnMuIiIiCgogICAgd2VpZ2h0czogbnAubmRhcnJheQogICAgbWVhbnM6IG5wLm5kYXJyYXkKICAgIGNvdmFyaWFuY2VzOiBucC5uZGFycmF5CgogICAgZGVmIF9fcG9zdF9pbml0X18oc2VsZikgLT4gTm9uZToKICAgICAgICB3ZWlnaHRzLCBtZWFucywgY292YXJpYW5jZXMgPSBtYXAobGFtYmRhIGE6IG5wLmFzYXJyYXkoYSwgZHR5cGU9bnAuZmxvYXQ2NCksIChzZWxmLndlaWdodHMsIHNlbGYubWVhbnMsIHNlbGYuY292YXJpYW5jZXMpKQogICAgICAgIGlmIG1lYW5zLm5kaW0gIT0gMyBvciB3ZWlnaHRzLnNoYXBlICE9IG1lYW5zLnNoYXBlWzoyXSBvciBjb3ZhcmlhbmNlcy5zaGFwZSAhPSBtZWFucy5zaGFwZVs6Ml0gKyBtZWFucy5zaGFwZVsyOl0gKiAyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCB3ZWlnaHRzIFtOLEtdLCBtZWFucyBbTixLLERdLCBjb3ZhcmlhbmNlcyBbTixLLEQsRF0iKQogICAgICAgIGlmIG5wLmFueSh3ZWlnaHRzIDwgMCkgb3Igbm90IG5wLmFsbGNsb3NlKHdlaWdodHMuc3VtKGF4aXM9MSksIDEuMCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1peHR1cmUgd2VpZ2h0cyBtdXN0IGJlIG5vbm5lZ2F0aXZlIGFuZCBzdW0gdG8gb25lIikKICAgICAgICBpZiBub3QgbnAuYWxsY2xvc2UoY292YXJpYW5jZXMsIG5wLnN3YXBheGVzKGNvdmFyaWFuY2VzLCAtMSwgLTIpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ292YXJpYW5jZXMgbXVzdCBiZSBzeW1tZXRyaWMiKQogICAgICAgIGlmIG5wLmFueShucC5saW5hbGcuZWlndmFsc2goY292YXJpYW5jZXMpIDw9IDApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDb3ZhcmlhbmNlcyBtdXN0IGJlIHBvc2l0aXZlIGRlZmluaXRlIikKCiAgICBAcHJvcGVydHkKICAgIGRlZiBtZWFuKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiUmV0dXJuIG1peHR1cmUgbWVhbnMgd2l0aCBzaGFwZSBgW04sRF1gLiIiIgogICAgICAgIHJldHVybiBucC5laW5zdW0oIm5rLG5rZC0+bmQiLCBzZWxmLndlaWdodHMsIHNlbGYubWVhbnMpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdG90YWxfY292YXJpYW5jZShzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgICIiIlJldHVybiBsYXctb2YtdG90YWwtdmFyaWFuY2UgY292YXJpYW5jZXMgd2l0aCBzaGFwZSBgW04sRCxEXWAuIiIiCiAgICAgICAgbWVhbiA9IHNlbGYubWVhbgogICAgICAgIGRlbHRhID0gc2VsZi5tZWFucyAtIG1lYW5bOiwgTm9uZSwgOl0KICAgICAgICByZXR1cm4gbnAuZWluc3VtKCJuayxua2RlLT5uZGUiLCBzZWxmLndlaWdodHMsIHNlbGYuY292YXJpYW5jZXMpICsgbnAuZWluc3VtKCJuayxua2QsbmtlLT5uZGUiLCBzZWxmLndlaWdodHMsIGRlbHRhLCBkZWx0YSkKCiAgICBkZWYgbG9ncGRmKHNlbGYsIHRhcmdldHM6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiRXZhbHVhdGUgbWl4dHVyZSBsb2cgZGVuc2l0aWVzIGF0IGBbTixEXWAgdGFyZ2V0cy4iIiIKICAgICAgICB5ID0gbnAuYXNhcnJheSh0YXJnZXRzLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgICAgIGlmIHkuc2hhcGUgIT0gc2VsZi5tZWFuLnNoYXBlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0YXJnZXRzIG11c3QgbWF0Y2ggcHJlZGljdGlvbiBtZWFuIHNoYXBlIikKICAgICAgICB0ZXJtcyA9IG5wLnN0YWNrKFtucC5sb2coc2VsZi53ZWlnaHRzWzosIGtdKSArIG5wLmFycmF5KFttdWx0aXZhcmlhdGVfbm9ybWFsLmxvZ3BkZih5W2ldLCBzZWxmLm1lYW5zW2ksIGtdLCBzZWxmLmNvdmFyaWFuY2VzW2ksIGtdKSBmb3IgaSBpbiByYW5nZSh5LnNoYXBlWzBdKV0pIGZvciBrIGluIHJhbmdlKHNlbGYud2VpZ2h0cy5zaGFwZVsxXSldLCBheGlzPTEpCiAgICAgICAgcmV0dXJuIGxvZ3N1bWV4cCh0ZXJtcywgYXhpcz0xKQoKICAgIGRlZiBjZGZfMWQoc2VsZiwgdGFyZ2V0czogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJFdmFsdWF0ZSAxLUQgbWl4dHVyZSBDREZzOyBtdWx0aXZhcmlhdGUgQ0RGIGlzIGludGVudGlvbmFsbHkgdW5zdXBwb3J0ZWQuIiIiCiAgICAgICAgaWYgc2VsZi5tZWFucy5zaGFwZVsyXSAhPSAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQSVQvQ0RGIGV2YWx1YXRpb24gaXMgZGVmaW5lZCBoZXJlIG9ubHkgZm9yIHVuaXZhcmlhdGUgdGFyZ2V0cyIpCiAgICAgICAgeSA9IG5wLmFzYXJyYXkodGFyZ2V0cywgZHR5cGU9bnAuZmxvYXQ2NCkucmVzaGFwZSgtMSkKICAgICAgICBzY2FsZXMgPSBucC5zcXJ0KHNlbGYuY292YXJpYW5jZXNbOiwgOiwgMCwgMF0pCiAgICAgICAgcmV0dXJuIG5wLnN1bShzZWxmLndlaWdodHMgKiBub3JtLmNkZigoeVs6LCBOb25lXSAtIHNlbGYubWVhbnNbOiwgOiwgMF0pIC8gc2NhbGVzKSwgYXhpcz0xKQoKCmRlZiBnYXVzc2lhbl9wcmVkaWN0aW9uKG1lYW46IG5wLm5kYXJyYXksIHZhcmlhbmNlOiBucC5uZGFycmF5LCBtaW5pbXVtX3ZhcmlhbmNlOiBmbG9hdCA9IDFlLTgpIC0+IEdhdXNzaWFuTWl4dHVyZVByZWRpY3Rpb246CiAgICAiIiJDcmVhdGUgYSBvbmUtY29tcG9uZW50IGRpYWdvbmFsIEdhdXNzaWFuIHByZWRpY3Rpb24gZnJvbSBtZWFucy92YXJpYW5jZXMuIiIiCiAgICBtdSA9IG5wLmFzYXJyYXkobWVhbiwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIHZhciA9IG5wLm1heGltdW0obnAuYXNhcnJheSh2YXJpYW5jZSwgZHR5cGU9bnAuZmxvYXQ2NCksIG1pbmltdW1fdmFyaWFuY2UpCiAgICBpZiBtdS5uZGltID09IDE6CiAgICAgICAgbXUsIHZhciA9IG11WzosIE5vbmVdLCB2YXJbOiwgTm9uZV0KICAgIGlmIG11LnNoYXBlICE9IHZhci5zaGFwZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtZWFuIGFuZCB2YXJpYW5jZSBtdXN0IGhhdmUgbWF0Y2hpbmcgc2hhcGVzIikKICAgIGNvdmFyaWFuY2UgPSBucC5leWUobXUuc2hhcGVbMV0pW05vbmUsIE5vbmVdICogdmFyWzosIE5vbmUsIDosIE5vbmVdCiAgICByZXR1cm4gR2F1c3NpYW5NaXh0dXJlUHJlZGljdGlvbihucC5vbmVzKChtdS5zaGFwZVswXSwgMSkpLCBtdVs6LCBOb25lLCA6XSwgY292YXJpYW5jZSkK", "src/plotting/__init__.py": "IiIiUHVibGljYXRpb24tb3JpZW50ZWQgdmlzdWFsaXphdGlvbiBoZWxwZXJzLiIiIgo=", "src/plotting/figures.py": "IiIiU2NhdHRlciwgaGVhdG1hcCwgZW1iZWRkaW5nLCBhbmQgY3JpdGljYWwtZGlmZmVyZW5jZS1zdHlsZSByYW5rIHBsb3RzLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHNlYWJvcm4gYXMgc25zCmZyb20gc2tsZWFybi5kZWNvbXBvc2l0aW9uIGltcG9ydCBQQ0EKZnJvbSBza2xlYXJuLm1hbmlmb2xkIGltcG9ydCBUU05FCgoKZGVmIHNhdmVfY29ycmVsYXRpb25fc2NhdHRlcihmcmFtZTogcGQuRGF0YUZyYW1lLCB4OiBzdHIsIHk6IHN0ciwgcGF0aDogc3RyIHwgUGF0aCkgLT4gUGF0aDoKICAgICIiIlNhdmUgYSBsYWJlbGxlZCBOUkMtZGlzdGFuY2UgY29ycmVsYXRpb24gc2NhdHRlciBwbG90LiIiIgogICAgZGVzdGluYXRpb24gPSBQYXRoKHBhdGgpOyBkZXN0aW5hdGlvbi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZmlndXJlLCBheGlzID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDYsIDQpKTsgc25zLnJlZ3Bsb3QoZGF0YT1mcmFtZSwgeD14LCB5PXksIGF4PWF4aXMsIHNjYXR0ZXJfa3dzPXsicyI6IDQ1fSkKICAgIGZvciByb3cgaW4gZnJhbWUuaXRlcnR1cGxlcygpOiBheGlzLmFubm90YXRlKHN0cihyb3dbMV0pLCAoZ2V0YXR0cihyb3csIHgpLCBnZXRhdHRyKHJvdywgeSkpLCBmb250c2l6ZT03KQogICAgZmlndXJlLnRpZ2h0X2xheW91dCgpOyBmaWd1cmUuc2F2ZWZpZyhkZXN0aW5hdGlvbiwgZHBpPTMwMCk7IHBsdC5jbG9zZShmaWd1cmUpOyByZXR1cm4gZGVzdGluYXRpb24KCgpkZWYgc2F2ZV9oZWF0bWFwKGZyYW1lOiBwZC5EYXRhRnJhbWUsIHBhdGg6IHN0ciB8IFBhdGgpIC0+IFBhdGg6CiAgICAiIiJTYXZlIGEgY29ycmVsYXRpb24gaGVhdG1hcCBmcm9tIG51bWVyaWMgcmVzdWx0IGNvbHVtbnMuIiIiCiAgICBkZXN0aW5hdGlvbiA9IFBhdGgocGF0aCk7IGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmaWd1cmUsIGF4aXMgPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNywgNSkpOyBzbnMuaGVhdG1hcChmcmFtZS5jb3JyKG51bWVyaWNfb25seT1UcnVlKSwgY21hcD0idmxhZyIsIGNlbnRlcj0wLCBhbm5vdD1UcnVlLCBmbXQ9Ii4yZiIsIGF4PWF4aXMpCiAgICBmaWd1cmUudGlnaHRfbGF5b3V0KCk7IGZpZ3VyZS5zYXZlZmlnKGRlc3RpbmF0aW9uLCBkcGk9MzAwKTsgcGx0LmNsb3NlKGZpZ3VyZSk7IHJldHVybiBkZXN0aW5hdGlvbgoKCmRlZiBlbWJlZGRpbmcoZmVhdHVyZXM6IG5wLm5kYXJyYXksIG1ldGhvZDogc3RyID0gInBjYSIsIHNlZWQ6IGludCA9IDApIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gMi1EIFBDQSBvciB0LVNORSBjb29yZGluYXRlczsgVU1BUCBpcyBvcHRpb25hbCBhdCBydW50aW1lLiIiIgogICAgbWF0cml4ID0gbnAuYXNhcnJheShmZWF0dXJlcywgZmxvYXQpCiAgICBpZiBtZXRob2QgPT0gInBjYSI6IHJldHVybiBQQ0Eobl9jb21wb25lbnRzPTIsIHJhbmRvbV9zdGF0ZT1zZWVkKS5maXRfdHJhbnNmb3JtKG1hdHJpeCkKICAgIGlmIG1ldGhvZCA9PSAidHNuZSI6IHJldHVybiBUU05FKG5fY29tcG9uZW50cz0yLCByYW5kb21fc3RhdGU9c2VlZCwgaW5pdD0icGNhIikuZml0X3RyYW5zZm9ybShtYXRyaXgpCiAgICBpZiBtZXRob2QgPT0gInVtYXAiOgogICAgICAgIGltcG9ydCB1bWFwCiAgICAgICAgcmV0dXJuIHVtYXAuVU1BUChuX2NvbXBvbmVudHM9MiwgcmFuZG9tX3N0YXRlPXNlZWQpLmZpdF90cmFuc2Zvcm0obWF0cml4KQogICAgcmFpc2UgVmFsdWVFcnJvcigibWV0aG9kIG11c3QgYmUgcGNhLCB0c25lLCBvciB1bWFwIikK", "src/utils/__init__.py": "IiIiUnVudGltZSwgSS9PLCBhbmQgbG9nZ2luZyBoZWxwZXJzIGZvciBOUkMtQ2FsIG5vdGVib29rcy4iIiIK", "src/utils/io.py": "IiIiQXRvbWljLCB0eXBlZCBhcnRpZmFjdCBwZXJzaXN0ZW5jZS4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgoKZGVmIHNhdmVfanNvbihwYXlsb2FkOiBkaWN0W3N0ciwgQW55XSwgcGF0aDogc3RyIHwgUGF0aCkgLT4gUGF0aDoKICAgICIiIkF0b21pY2FsbHkgd3JpdGUgSlNPTiBhbmQgcmV0dXJuIGl0cyBub3JtYWxpemVkIHBhdGguIiIiCiAgICBkZXN0aW5hdGlvbiA9IFBhdGgocGF0aCkKICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0ZW1wb3JhcnkgPSBkZXN0aW5hdGlvbi53aXRoX3N1ZmZpeChkZXN0aW5hdGlvbi5zdWZmaXggKyAiLnRtcCIpCiAgICB0ZW1wb3Jhcnkud3JpdGVfdGV4dChqc29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSwgZGVmYXVsdD1zdHIpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHRlbXBvcmFyeS5yZXBsYWNlKGRlc3RpbmF0aW9uKQogICAgcmV0dXJuIGRlc3RpbmF0aW9uCgoKZGVmIHNhdmVfZnJhbWUoZnJhbWU6IHBkLkRhdGFGcmFtZSwgcGF0aDogc3RyIHwgUGF0aCkgLT4gUGF0aDoKICAgICIiIlNhdmUgYSBDU1YgcmVzdWx0IHRhYmxlIHdpdGggYSBzdGFibGUgc2NoZW1hIGFuZCBubyBpbmRleC4iIiIKICAgIGRlc3RpbmF0aW9uID0gUGF0aChwYXRoKQogICAgZGVzdGluYXRpb24ucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGZyYW1lLnRvX2NzdihkZXN0aW5hdGlvbiwgaW5kZXg9RmFsc2UpCiAgICByZXR1cm4gZGVzdGluYXRpb24KCgpkZWYgc2F2ZV9hcnJheXMocGF0aDogc3RyIHwgUGF0aCwgKiphcnJheXM6IG5wLm5kYXJyYXkpIC0+IFBhdGg6CiAgICAiIiJQZXJzaXN0IG5hbWVkIGFycmF5cyBpbiBhIGNvbXByZXNzZWQgTlBaIGZlYXR1cmUgY2FjaGUuIiIiCiAgICBkZXN0aW5hdGlvbiA9IFBhdGgocGF0aCkKICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBucC5zYXZlel9jb21wcmVzc2VkKGRlc3RpbmF0aW9uLCAqKmFycmF5cykKICAgIHJldHVybiBkZXN0aW5hdGlvbgo=", "src/utils/runtime.py": "IiIiQ29sYWItYXdhcmUgcnVudGltZSBpbml0aWFsaXphdGlvbiB1c2VkIGF0IHRoZSBzdGFydCBvZiBldmVyeSBub3RlYm9vay4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0LCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUnVudGltZUluZm86CiAgICAiIiJSZXNvbHZlZCBwcm9qZWN0IGFuZCBoYXJkd2FyZSBjb25maWd1cmF0aW9uLiIiIgoKICAgIHByb2plY3Rfcm9vdDogUGF0aAogICAgY2hlY2twb2ludF9yb290OiBQYXRoCiAgICBjdWRhX2F2YWlsYWJsZTogYm9vbAogICAgdG9yY2hfdmVyc2lvbjogc3RyCiAgICBjdWRhX3ZlcnNpb246IHN0ciB8IE5vbmUKICAgIHJhbV9naWI6IGZsb2F0CiAgICBncHVfbmFtZTogc3RyIHwgTm9uZQoKCmRlZiBjb25maWd1cmVfcnVudGltZShwcm9qZWN0X3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSkgLT4gUnVudGltZUluZm86CiAgICAiIiJSZXNvbHZlIHRoZSB1cGxvYWRlZC9sb2NhbCBwcm9qZWN0IHJvb3QgYW5kIGVuYWJsZSBzYWZlIENVREEgc2V0dGluZ3MuIiIiCiAgICBpbXBvcnQgcHN1dGlsCiAgICBpbXBvcnQgdG9yY2gKCiAgICBjb25maWd1cmVkID0gcHJvamVjdF9yb290IG9yIG9zLmVudmlyb24uZ2V0KCJOUkNfQ0FMX1BST0pFQ1RfUk9PVCIpCiAgICBjYW5kaWRhdGVzID0gW1BhdGgoY29uZmlndXJlZCkuZXhwYW5kdXNlcigpXSBpZiBjb25maWd1cmVkIGVsc2UgW10KICAgIGNhbmRpZGF0ZXMuZXh0ZW5kKFtQYXRoKCIvY29udGVudC9OUkNfQ0FMSUJfQ09ERSIpLCBQYXRoLmN3ZCgpLnBhcmVudCwgUGF0aC5jd2QoKV0pCiAgICByb290ID0gbmV4dCgocGF0aCBmb3IgcGF0aCBpbiBjYW5kaWRhdGVzIGlmIChwYXRoIC8gInNyYyIpLmlzX2RpcigpKSwgTm9uZSkKICAgIGlmIHJvb3QgaXMgTm9uZToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgIk5SQy1DYWwgcHJvamVjdCBub3QgZm91bmQuIFVwbG9hZCB0aGUgcHJvamVjdCBaSVAgaW4gbm90ZWJvb2sgMDAgb3Igc2V0ICIKICAgICAgICAgICAgIk5SQ19DQUxfUFJPSkVDVF9ST09UIHRvIGEgZGlyZWN0b3J5IGNvbnRhaW5pbmcgc3JjLy4iCiAgICAgICAgKQogICAgcm9vdCA9IHJvb3QucmVzb2x2ZSgpCiAgICBmb3IgbmFtZSBpbiAoIm91dHB1dHMiLCAiZmlndXJlcyIsICJjaGVja3BvaW50cyIsICJsb2dzIik6CiAgICAgICAgKHJvb3QgLyBuYW1lKS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjaGVja3BvaW50X3Jvb3QgPSByb290IC8gImNoZWNrcG9pbnRzIgogICAgY2hlY2twb2ludF9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG9zLmVudmlyb25bIk5SQ19DQUxfUFJPSkVDVF9ST09UIl0gPSBzdHIocm9vdCkKICAgIG9zLmVudmlyb25bIk5SQ19DQUxfQ0hFQ0tQT0lOVF9ST09UIl0gPSBzdHIoY2hlY2twb2ludF9yb290KQogICAgY3VkYSA9IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkKICAgIGdwdV9uYW1lID0gTm9uZQogICAgaWYgY3VkYToKICAgICAgICB0b3JjaC5iYWNrZW5kcy5jdWRhLm1hdG11bC5hbGxvd190ZjMyID0gVHJ1ZQogICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmFsbG93X3RmMzIgPSBUcnVlCiAgICAgICAgdG9yY2guc2V0X2Zsb2F0MzJfbWF0bXVsX3ByZWNpc2lvbigiaGlnaCIpCiAgICAgICAgZ3B1X25hbWUgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKQogICAgaW5mbyA9IFJ1bnRpbWVJbmZvKHJvb3QsIGNoZWNrcG9pbnRfcm9vdCwgY3VkYSwgdG9yY2guX192ZXJzaW9uX18sIHRvcmNoLnZlcnNpb24uY3VkYSwKICAgICAgICAgICAgICAgICAgICAgICBwc3V0aWwudmlydHVhbF9tZW1vcnkoKS50b3RhbCAvIDIqKjMwLCBncHVfbmFtZSkKICAgIGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKS5pbmZvKCJSdW50aW1lOiAlcyIsIGFzZGljdChpbmZvKSkKICAgIHJldHVybiBpbmZvCgoKZGVmIGFkZF9wcm9qZWN0X3NvdXJjZShwcm9qZWN0X3Jvb3Q6IFBhdGgpIC0+IE5vbmU6CiAgICAiIiJQbGFjZSB0aGUgcHJvamVjdCdzIGhlbHBlciBtb2R1bGVzIGZpcnN0IG9uIFB5dGhvbidzIGltcG9ydCBwYXRoLiIiIgogICAgc291cmNlID0gc3RyKHByb2plY3Rfcm9vdCAvICJzcmMiKQogICAgaWYgc291cmNlIG5vdCBpbiBzeXMucGF0aDoKICAgICAgICBzeXMucGF0aC5pbnNlcnQoMCwgc291cmNlKQoKCmRlZiBncHVfc3VtbWFyeSgpIC0+IHN0cjoKICAgICIiIlJldHVybiBhIGNvbXBhY3QgQ1VEQS1kcml2ZXIgc3VtbWFyeSB3aXRob3V0IGZhaWxpbmcgb24gQ1BVIGhvc3RzLiIiIgogICAgaWYgc2h1dGlsLndoaWNoKCJudmlkaWEtc21pIikgaXMgTm9uZToKICAgICAgICByZXR1cm4gIm52aWRpYS1zbWkgdW5hdmFpbGFibGUiCiAgICByZXR1cm4gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgWyJudmlkaWEtc21pIiwgIi0tcXVlcnktZ3B1PW5hbWUsZHJpdmVyX3ZlcnNpb24sbWVtb3J5LnRvdGFsIiwgIi0tZm9ybWF0PWNzdixub2hlYWRlciJdLAogICAgICAgIHRleHQ9VHJ1ZSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9RmFsc2UsCiAgICApLnN0ZG91dC5zdHJpcCgpCg==", "docs/methodology.md": "IyBOUkMtQ2FsIE1ldGhvZG9sb2d5IGFuZCBBdHRyaWJ1dGlvbgoKVGhpcyBkb2N1bWVudCBpcyB0aGUgbm9ybWF0aXZlIG1ldGhvZHMgc3BlY2lmaWNhdGlvbi4gRXF1YXRpb25zIGxhYmVsbGVkCioqUHVibGlzaGVkIE5SQyB0aGVvcnkqKiBhcmUgdHJhbnNjcmliZWQgZnJvbSBBbmRyaW9wb3Vsb3MgZXQgYWwuLCAqVGhlClByZXZhbGVuY2Ugb2YgTmV1cmFsIENvbGxhcHNlIGluIE5ldXJhbCBNdWx0aXZhcmlhdGUgUmVncmVzc2lvbiosIE5ldXJJUFMKMjAyNC4gRXF1YXRpb25zIGxhYmVsbGVkICoqTlJDLUNhbCBwcm9wb3NhbCoqIGFyZSBvcmlnaW5hbCBtZXRob2RvbG9neSBpbiB0aGlzCnJlcG9zaXRvcnksIG5vdCByZXN1bHRzIG9yIGNsYWltcyBvZiBlaXRoZXIgTlJDIHBhcGVyLgoKIyMgTm90YXRpb24KCkZvciBgTWAgZXhhbXBsZXMsIGB4X2lgIGlzIGlucHV0IGBpYCwgYHlfaSBpbiBSXm5gIGl0cyB0YXJnZXQsIGFuZApgaF9pIGluIFJeZGAgaXRzIGxhc3QtaGlkZGVuLWxheWVyIGZlYXR1cmUuIGBIPVtoXzEsLi4uLGhfTV0gaW4gUl4oZCB4IE0pYC4KYFcgaW4gUl4obiB4IGQpYCBpcyB0aGUgZmluYWwgYWZmaW5lIG1lYW4taGVhZCB3ZWlnaHQgbWF0cml4LiBgfHwufHxfMmAgYW5kCmB8fC58fF9GYCBhcmUgRXVjbGlkZWFuIGFuZCBGcm9iZW5pdXMgbm9ybXMuIGBQX0ModilgIGRlbm90ZXMgb3J0aG9nb25hbApwcm9qZWN0aW9uIG9mIGB2YCBvbnRvIHRoZSBjb2x1bW4gc3BhbiBvZiBgQ2AuIGBTaWdtYWAgaXMgdGhlIHBvcHVsYXRpb24tZm9ybQplbXBpcmljYWwgdGFyZ2V0IGNvdmFyaWFuY2UgYE1eLTEgKFktWWJhcikoWS1ZYmFyKV5UYCwgYW5kIGBsYW1iZGFfbWluYCBpcwppdHMgc21hbGxlc3QgZWlnZW52YWx1ZS4gYElfbmAgaXMgdGhlIGBuIHggbmAgaWRlbnRpdHkgbWF0cml4LgoKIyMgUHVibGlzaGVkIE5SQyB0aGVvcnkgKHZlcmJhdGltIG1hdGhlbWF0aWNhbCBjb250ZW50KQoKVGhlIE5ldXJJUFMgMjAyNCBwYXBlciBkZWZpbmVzIGB0aWxkZShoX2kpPWhfaSB8fGhfaXx8XzJeLTFgIGFuZCBsZXRzCmBIX1BDQV9uYCBjb250YWluIHRoZSBmaXJzdCBgbmAgcHJpbmNpcGFsIGNvbXBvbmVudHMgb2YgYEhgLiBOUkMgZW1lcmdlcyB3aGVuCgpgYGAKTlJDMSA9IE1eLTEgc3VtX2kgfHx0aWxkZShoX2kpIC0gUF97SF9QQ0Ffbn0odGlsZGUoaF9pKSl8fF8yXjIgLT4gMCwKTlJDMiA9IE1eLTEgc3VtX2kgfHx0aWxkZShoX2kpIC0gUF97V15UfSh0aWxkZShoX2kpKXx8XzJeMiAtPiAwLApOUkMzID0gfHwgV1deVC98fFdXXlR8fF9GCiAgICAgICAtIChTaWdtYV4oMS8yKS1nYW1tYV4oMS8yKUlfbikvfHxTaWdtYV4oMS8yKS1nYW1tYV4oMS8yKUlfbnx8X0YgfHxfRl4yIC0+IDAsCmBgYAoKZm9yIGEgY29uc3RhbnQgYGdhbW1hIGluICgwLCBsYW1iZGFfbWluKWAuIFRoZSBwYXBlciBzcGVjaWZpZXMgdGhhdCBpbiB0aGUKdW5pdmFyaWF0ZSBjYXNlIE5SQzMgaXMgdHJpdmlhbGx5IHplcm8uIEl0IGZpbmRzIGBnYW1tYWAgYnkgbWluaW1pemluZyBOUkMzCmZvciB0aGUgZmluYWwgdHJhaW5lZCB3ZWlnaHQgbWF0cml4LiBUaGUgMjAyNSBpbnRyaW5zaWMtZGltZW5zaW9uIHBhcGVyIHVzZXMKYSBkaXN0aW5jdCAqY2VudGVyZWQtbm9ybWFsaXplZCogTlJDMSB2YXJpYW50LCB3aXRoCmB0aWxkZShoX2kpPShoX2ktaGJhcikvfHxoX2ktaGJhcnx8XzJgOyBpdCBpcyBleHBvc2VkIGFzIGFuIGV4cGxpY2l0bHkgbmFtZWQKYWx0ZXJuYXRpdmUsIG5ldmVyIHNpbGVudGx5IHN1YnN0aXR1dGVkIGZvciB0aGUgMjAyNCBkZWZpbml0aW9uLgoKU291cmNlIHNuYXBzaG90czogYHJlZmVyZW5jZXMvbnJjX25ldXJpcHNfMjAyNC9uZXVyaXBzXzIwMjQudGV4OjI0My0yODVgIGFuZApgcmVmZXJlbmNlcy9ucmNfaW50cmluc2ljX2RpbWVuc2lvbl8yMDI1L25ldXJpcHNfMjAyNi50ZXg6MjUzLTI3OGAuCgojIyBOUkMtQ2FsIHByb3Bvc2FsOiBkaXN0YW5jZXMKCioqTmV3IGFzc3VtcHRpb24gQTEgKGZyb3plbiByZXByZXNlbnRhdGlvbikuKiogRmVhdHVyZSB2ZWN0b3JzIGFuZCBmaW5hbAptZWFuLWhlYWQgd2VpZ2h0cyBhcmUgbWVhc3VyZWQgYWZ0ZXIgdHJhaW5pbmcgYW5kIHJlbWFpbiBmaXhlZC4gKipBMgoobm9uemVybyBmZWF0dXJlcykuKiogRXZlcnkgcmV0YWluZWQgZmVhdHVyZSBoYXMgbm9ybSBncmVhdGVyIHRoYW4gZXBzaWxvbi4KCkRlZmluZSBwdWJsaXNoZWQgcGVyLWV4YW1wbGUgcmVzaWR1YWxzCmByMV9pPXx8dGlsZGUoaF9pKS1QX3tIX1BDQV9ufSh0aWxkZShoX2kpKXx8XzJeMmAgYW5kCmByMl9pPXx8dGlsZGUoaF9pKS1QX3tXXlR9KHRpbGRlKGhfaSkpfHxfMl4yYC4gTGV0IGBtMT1NXi0xIHN1bSByMV9pYCwKYG0yPU1eLTEgc3VtIHIyX2lgLCBhbmQgYG0zPU5SQzNgLiBMZXQgYHc9KHcxLHcyLHczKWAgc2F0aXNmeSBgd19qPj0wYCBhbmQKYHN1bV9qIHdfaj0xYC4gRm9yIHVuaXZhcmlhdGUgdGFyZ2V0cyB3ZSBzZXQgYHczPTBgIGFuZCByZW5vcm1hbGl6ZSBgKHcxLHcyKWAKYmVjYXVzZSB0aGUgcHVibGlzaGVkIHBhcGVyIHNheXMgTlJDMyBpcyB0cml2aWFsLgoKKipEYXRhc2V0LWxldmVsIE5SQy1kaXN0YW5jZSAobmV3KToqKgpgRF9kYXRhc2V0ID0gdzEgbTEgKyB3MiBtMiArIHczIG0zYC4KCioqU2FtcGxlLWxldmVsIE5SQy1kaXN0YW5jZSAobmV3KToqKgpgRF9pID0gdzEgcjFfaSArIHcyIHIyX2kgKyB3MyBtM2AuCgpNb3RpdmF0aW9uOiBlYWNoIHRlcm0gaXMgYW4gZXhpc3RpbmcgYm91bmRlZCBzcXVhcmVkIGdlb21ldHJpYyByZXNpZHVhbDsKY29udmV4IGFnZ3JlZ2F0aW9uIGFkZHMgbm8gdW5jYWxpYnJhdGVkIHNjYWxlIG9yIGxlYXJuZWQgcmVwcmVzZW50YXRpb24uClRoZSBrZXkgcHJvcGVydHkgaXMgZXhhY3QgY29uc2lzdGVuY3k6IGBNXi0xIHN1bV9pIERfaSA9IERfZGF0YXNldGAuCldpdGggbm9ybWFsaXplZCBpbnB1dHMsIGByMV9pLHIyX2kgaW4gWzAsMV1gIGFuZCBgbTMgaW4gWzAsNF1gLCBoZW5jZQpgRF9pLERfZGF0YXNldCBpbiBbMCw0XWAuIFBDQSBjb3N0cyBgTyhNIGQgbWluKE0sZCkpYDsgcHJvamVjdGlvbiByZXNpZHVhbHMKY29zdCBgTyhNZG4pYCB1c2luZyB0aGluIG9ydGhvbm9ybWFsIGJhc2VzOyBOUkMzIGNvc3RzIGBPKG5eMytLIG5eMilgIGZvciBgS2AKY2FuZGlkYXRlIGdhbW1hIGV2YWx1YXRpb25zLiBFeHBlY3RlZCBiZWhhdmlvcjogbGFyZ2UgdmFsdWVzIGZsYWcgZGVwYXJ0dXJlCmZyb20gdGhlIHRocmVlIGNvbGxhcHNlIHJlbGF0aW9uc2hpcHMsIG5vdCBjYWxpYnJhdGlvbiBlcnJvciBpdHNlbGYuCgojIyBOUkMtQ2FsIHByb3Bvc2FsOiBjbG9zZWQtZm9ybSBjYWxpYnJhdGlvbiBtYXAKCkxldCBhbiBhcmJpdHJhcnkgZnJvemVuIEdhdXNzaWFuIG9yIGBLYC1jb21wb25lbnQgR2F1c3NpYW4gbWl4dHVyZSB5aWVsZCBtZWFuCmBtdV9pIGluIFJebmAgYW5kIHRvdGFsIGNvdmFyaWFuY2UgYFZfaWAgKHN0cmljdGx5IHBvc2l0aXZlIGRlZmluaXRlKS4gRGVmaW5lCnRoZSBzcXVhcmVkIE1haGFsYW5vYmlzIHJlc2lkdWFsIGBxX2k9KHlfaS1tdV9pKV5UIFZfaV4tMSh5X2ktbXVfaSkvbmAgYW5kCnRoZSBzdGFuZGFyZGl6ZWQgZ2VvbWV0cnkgY292YXJpYXRlIGB6X2k9KERfaS1EYmFyKS8oc19EK2Vwc2lsb24pYCwgd2hlcmUKYERiYXI9TV4tMSBzdW1faSBEX2lgIGFuZCBgc19EPVtNXi0xIHN1bV9pKERfaS1EYmFyKV4yXV4oMS8yKWAuCgoqKk5ldyBhc3N1bXB0aW9uIEEzIChsb2ctc2NhbGUgbW9kZWwpLioqIE9uIGFuIGluZGVwZW5kZW50IGNhbGlicmF0aW9uIHNwbGl0LApgbG9nKHFfaSk9YStiIHpfaStldGFfaWAsIHdpdGggZmluaXRlLXZhcmlhbmNlLCBtZWFuLXplcm8gcmVzaWR1YWwgYGV0YV9pYC4KRm9yIGFuIGV4YWN0bHkgY2FsaWJyYXRlZCBHYXVzc2lhbiwgYG4gcV9pYCBpcyBjaGktc3F1YXJlIHdpdGggYG5gIGRlZ3JlZXMgb2YKZnJlZWRvbS4gSXRzIGV4cGVjdGVkIGxvZyBzY2FsZSBpcwpgdGF1X249cHNpKG4vMikrbG9nKDIpLWxvZyhuKWAsIHdoZXJlIGBwc2lgIGlzIHRoZSBkaWdhbW1hIGZ1bmN0aW9uLgoKV2l0aCBgWD1bMSx6XWAsIGRlZmluZSB0aGUgcmlkZ2Utc3RhYmlsaXplZCwgY2xvc2VkLWZvcm0gZXN0aW1hdGUKYHRoZXRhPShhLGIpXlQ9KFheVCBYICsgbGFtYmRhIGRpYWcoMCwxKSleLTEgWF5UIGxvZyhxKWAuIEhlcmUgYGxhbWJkYT49MGAKaXMgYSBmaXhlZCBzdGFiaWxpdHkgcGVuYWx0eSwgY2hvc2VuIGJ5IGEgZGV0ZXJtaW5pc3RpYyBjYWxpYnJhdGlvbi1vbmx5IGdyaWQKc2VhcmNoIG1pbmltaXppbmcgUENFOyBgbGFtYmRhPTBgIGlzIG9yZGluYXJ5IGxlYXN0IHNxdWFyZXMgd2hlbmV2ZXIgdGhlCm1hdHJpeCBpcyBub25zaW5ndWxhci4gVGhlIGNvcnJlY3Rpb24gaXMKYHNfaT1jbGlwKGV4cCgoYStiIHpfaS10YXVfbikvMiksIHNfbWluLCBzX21heClgLgoKRm9yIGEgR2F1c3NpYW4sIE5SQy1DYWwgbWFwcyBgKG11X2ksVl9pKWAgdG8gYChtdV9pLHNfaV4yIFZfaSlgLiBGb3IgYQptaXh0dXJlIHdpdGggY29tcG9uZW50IHdlaWdodHMgYHBpX2lrYCwgbWVhbnMgYG11X2lrYCwgYW5kIGNvdmFyaWFuY2VzCmBWX2lrYCwgd3JpdGUgYG11X2k9c3VtX2sgcGlfaWsgbXVfaWtgIGFuZCBtYXAKYG11J19paz1tdV9pK3NfaShtdV9pay1tdV9pKWAsIGBWJ19paz1zX2leMiBWX2lrYCwgcHJlc2VydmluZyBgcGlfaWtgLgpUaGlzIGlzIGEgdmFsaWQgR2F1c3NpYW4gbWl4dHVyZSBhbmQgZXhhY3RseSBtYXBzIGl0cyB0b3RhbCBjb3ZhcmlhbmNlIHRvCmBzX2leMiBWX2lgOyB0aGVyZWZvcmUgaXQgd29ya3Mgd2l0aG91dCByZXRyYWluaW5nIG9yIGNoYW5naW5nIG1peHR1cmUgbWFzcy4KCioqUHJvcG9zaXRpb24gKG5ldzsgYXNzdW1wdGlvbnMgQTEtLUEzIGFuZCBubyBjbGlwcGluZykuKiogVGhlIG1hcCBtYWtlcyB0aGUKZml0dGVkIGNvbmRpdGlvbmFsIGxvZy1NYWhhbGFub2JpcyByZXNpZHVhbCBpbmRlcGVuZGVudCBvZiBgemAgaW4gdGhlIGxpbmVhcgptb2RlbDogYGxvZyhxX2kvc19pXjIpPXRhdV9uK2V0YV9pYC4gUHJvb2Y6IHN1YnN0aXR1dGUgdGhlIGRlZmluaXRpb24gb2YKYHNfaV4yYCBpbnRvIGBsb2cgcV9pIC0gbG9nIHNfaV4yYC4gVGh1cyB0aGUgZml0dGVkIGdlb21ldHJ5LWRlcGVuZGVudCBzY2FsZQp0cmVuZCBpcyByZW1vdmVkIHdoaWxlIGxvY2F0aW9uIGFuZCBtaXh0dXJlIHdlaWdodHMgcmVtYWluIHVuY2hhbmdlZC4KCioqU3RhYmlsaXR5LioqIFJpZGdlIG1ha2VzIGBYXlQgWCArIGxhbWJkYSBkaWFnKDAsMSlgIGludmVydGlibGUgd2hlbmV2ZXIgdGhlCmludGVyY2VwdCBjb2x1bW4gaXMgcHJlc2VudCBhbmQgYGxhbWJkYT4wYDsgY2xpcHBpbmcgZ2l2ZXMKYHNfaSBpbiBbc19taW4sc19tYXhdYCwgc28gdHJhbnNmb3JtZWQgY292YXJpYW5jZSBlaWdlbnZhbHVlcyBsaWUgaW4KYFtzX21pbl4yIGxhbWJkYV9taW4oVl9pKSwgc19tYXheMiBsYW1iZGFfbWF4KFZfaSldYC4gRml0IGNvbXBsZXhpdHkgaXMKYE8oTSlgIGZvciBhIHR3by1jb2x1bW4gcmVncmVzc2lvbiBhZnRlciBgcWA7IEdhdXNzaWFuIGV2YWx1YXRpb24gaXMKYE8oTSBuXjMpYCB3aXRoIENob2xlc2t5IHNvbHZlcyBhbmQgbWl4dHVyZSBtb21lbnRzIGFyZSBgTyhNIEsgbl4yKWAuCgojIyBBbGdvcml0aG1zIGFuZCBhYmxhdGlvbnMKCioqQWxnb3JpdGhtIDEgKG5ldyBOUkMtQ2FsIGRpc3RhbmNlLCBwc2V1ZG9jb2RlKS4qKiBJbnB1dDogZnJvemVuIGBILFksV2AgYW5kCmNvbnZleCBgd2AuICgxKSBOb3JtYWxpemUgZWFjaCBmZWF0dXJlIGV4YWN0bHkgYWNjb3JkaW5nIHRvIHRoZSBzZWxlY3RlZApjaXRlZCBwYXBlci4gKDIpIENvbXB1dGUgYHIxX2lgIGJ5IHByb2plY3Rpb24gb250byB0b3AtYG5gIFBDQSBjb2x1bW5zIGFuZApgcjJfaWAgYnkgcHJvamVjdGlvbiBvbnRvIGBjb2woV15UKWAuICgzKSBNaW5pbWl6ZSB0aGUgcHVibGlzaGVkIE5SQzMKb2JqZWN0aXZlIG92ZXIgYGdhbW1hIGluICgwLGxhbWJkYV9taW4pYDsgc2V0IGl0IHRvIHplcm8gb25seSBmb3IgYG49MWAsIGFzCnNwZWNpZmllZCBieSB0aGUgc291cmNlLiAoNCkgUmVub3JtYWxpemUgYHcxLHcyYCB3aGVuIGBuPTFgLiAoNSkgUmV0dXJuCmBEX2k9dzEgcjFfaSt3MiByMl9pK3czIE5SQzNgIGFuZCBpdHMgbWVhbiBgRF9kYXRhc2V0YC4KCioqQWxnb3JpdGhtIDIgKG5ldyBOUkMtQ2FsIG1hcCwgcHNldWRvY29kZSkuKiogSW5wdXQ6IGNhbGlicmF0aW9uIHRhcmdldHMsCmZyb3plbiBHYXVzc2lhbi9taXh0dXJlIHByZWRpY3Rpb24sIGFuZCBjYWxpYnJhdGlvbiBgRF9pYC4gKDEpIENhbGN1bGF0ZQp0b3RhbCBtaXh0dXJlIGNvdmFyaWFuY2UgYW5kIGBxX2lgLiAoMikgU3RhbmRhcmRpemUgYERfaWAgaW50byBgel9pYC4gKDMpCkZvciBlYWNoIGZpeGVkIHJpZGdlIGNhbmRpZGF0ZSwgc29sdmUgdGhlIGRpc3BsYXllZCB0d28tYnktdHdvIG5vcm1hbAplcXVhdGlvbnMgYW5kIHNlbGVjdCB0aGUgY2FsaWJyYXRpb24tb25seSBQQ0UgbWluaW1pemVyLiAoNCkgQ29tcHV0ZSBjbGlwcGVkCnBvc2l0aXZlIHNjYWxlcy4gKDUpIEF0IGluZmVyZW5jZSwgdHJhbnNmb3JtIGV2ZXJ5IG1peHR1cmUgY29tcG9uZW50IHdpdGgKdGhlIGRpc3BsYXllZCBtZWFuL2NvdmFyaWFuY2UgZXF1YXRpb25zLCB0aGVuIGV2YWx1YXRlIHVuY2hhbmdlZCBmcm96ZW4KbWVhbnMgYW5kIHRyYW5zZm9ybWVkIHVuY2VydGFpbnR5LgoKVGhlIGNvZGUgaW1wbGVtZW50YXRpb25zIGFyZSBgc3JjL2dlb21ldHJ5L25yYy5weWAgYW5kCmBzcmMvY2FsaWJyYXRpb24vbnJjX2NhbC5weWA7IHRoZWlyIHVuaXQgdGVzdHMgdmVyaWZ5IHRoZSBtZWFuLWRpc3RhbmNlIGFuZAptaXh0dXJlLW1lYW4vcG9zaXRpdmUtZGVmaW5pdGVuZXNzIGludmFyaWFudHMuCgpBYmxhdGlvbnMgYXJlOiBgRD1OUkMxYCwgYEQ9TlJDMmAsIGBEPU5SQzNgIChtdWx0aXZhcmlhdGUgb25seSksIGVxdWFsLXdlaWdodApmdWxsIGRpc3RhbmNlLCBubyBnZW9tZXRyeSAoYHo9MGAsIGdsb2JhbCBzY2FsZSksIG5vIHJpZGdlLCBhbmQgbm8gY2xpcHBpbmcuCkFsbCBhYmxhdGlvbnMgcmV0YWluIHRoZSBzYW1lIGZyb3plbiBmZWF0dXJlcywgY2FsaWJyYXRpb24gc3BsaXQsIGFuZCBtZXRyaWNzLgo=", "configs/default.yaml": "c2VlZDogMjAyNApwcm9qZWN0X25hbWU6IE5SQy1DYWwKZGF0YToKICBxcnRfbWFuaWZlc3Rfc2l6ZTogNTcKICBzcGxpdF9yYXRpb3M6IFswLjY1LCAwLjEwLCAwLjE1LCAwLjEwXQogIGNhbGlicmF0aW9uX2ZyYWN0aW9uOiAwLjE1Cmdlb21ldHJ5OgogIG5vcm1hbGl6YXRpb246IG5ldXJpcHNfMjAyNAogIHRhcmdldF9zdWJzcGFjZTogdGFyZ2V0X2RpbWVuc2lvbgogIHdlaWdodHM6IFswLjMzMzMzMzMzMzMzMzMzMzMsIDAuMzMzMzMzMzMzMzMzMzMzMywgMC4zMzMzMzMzMzMzMzMzMzMzXQogIGVwc2lsb246IDEuMGUtMTIKY2FsaWJyYXRpb246CiAgcmlkZ2VfY2FuZGlkYXRlczogWzAuMCwgMS4wZS04LCAxLjBlLTYsIDEuMGUtNCwgMS4wZS0yXQogIHNjYWxlX21pbjogMC4yNQogIHNjYWxlX21heDogNC4wCmV2YWx1YXRpb246CiAgcGNlX2xldmVsczogMTAwCiAgYm9vdHN0cmFwX3NhbXBsZXM6IDIwMDAKICBwZXJtdXRhdGlvbl9zYW1wbGVzOiA1MDAwCg=="}
for relative, encoded in SOURCE_FILES.items():
    destination = PROJECT_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
source_root = str(PROJECT_ROOT / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)
for module_name in ("geometry.nrc", "calibration.nrc_cal", "models.predictions", "metrics.evaluation", "metrics.statistics"):
    importlib.import_module(module_name)
print(f"Embedded and verified {len(SOURCE_FILES)} source files.")


## 4. Hardware and reproducibility record


In [ ]:
import datetime, psutil, subprocess, sys, torch
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
record = {"created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda, "cuda_available": torch.cuda.is_available(), "gpu_count": torch.cuda.device_count(), "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "ram_gib": psutil.virtual_memory().total / 2**30, "seed": SEED}
(PROJECT_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs/environment.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
(PROJECT_ROOT / "outputs/requirements-freeze.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout, encoding="utf-8")
print(json.dumps(record, indent=2))


## 5. Load the Kaggle regression task and build a frozen Gaussian baseline

The notebook uses `minhqunhc/aic-2026` as its primary source. It discovers tabular files, infers or accepts an explicit numeric target, and creates one task per eligible file or optional `TASK_COLUMN` group. It splits before train-only preprocessing and automatically uses the T4 GPU when available.

If Kaggle returns `403`, add a Colab secret named `KAGGLE_API_TOKEN` or upload `kaggle.json` to the standard Kaggle config location, then rerun this cell.


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

KAGGLE_HANDLE = "minhqunhc/aic-2026"
KAGGLE_FILE_PATH = ""       # Set e.g. "train.csv" after discovery if needed.
TARGET_COLUMN = ""          # Set explicitly if automatic inference chooses incorrectly.
TASK_COLUMN = ""             # Optional independent-task/group column.
DROP_COLUMNS = []
MAX_ROWS_PER_TASK = 120_000
MIN_ROWS_PER_TASK = 128

def _read_tabular(path):
    suffix = path.suffix.lower()
    if suffix == ".csv": return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}: return pd.read_parquet(path)
    if suffix in {".xlsx", ".xls"}: return pd.read_excel(path)
    if suffix == ".json": return pd.read_json(path)
    raise ValueError(f"Unsupported tabular suffix: {path.suffix}")

def _infer_target(frame):
    if TARGET_COLUMN and TARGET_COLUMN in frame.columns: return TARGET_COLUMN
    common = ("target", "label", "y", "score", "price", "sales", "output", "response")
    lower = {str(column).lower(): column for column in frame.columns}
    for name in common:
        if name in lower and pd.api.types.is_numeric_dtype(frame[lower[name]]): return lower[name]
    numeric = frame.select_dtypes(include=np.number).columns.tolist()
    return numeric[-1] if len(numeric) >= 2 else None

def load_kaggle_tasks():
    import kagglehub
    try:
        local_input = Path("/kaggle/input/aic-2026")
        dataset_root = local_input if local_input.is_dir() else Path(kagglehub.dataset_download(KAGGLE_HANDLE))
    except Exception as error:
        raise RuntimeError("Kaggle download failed. Add KAGGLE_API_TOKEN as a Colab secret or upload /root/.config/kaggle/kaggle.json, then rerun this cell. Original error: " + str(error)) from error
    if KAGGLE_FILE_PATH:
        candidate = dataset_root / KAGGLE_FILE_PATH
        if not candidate.is_file(): raise FileNotFoundError(f"KAGGLE_FILE_PATH not found: {candidate}")
        files = [candidate]
    else:
        files = sorted(path for path in dataset_root.rglob("*") if path.suffix.lower() in {".csv", ".parquet", ".pq", ".xlsx", ".xls", ".json"})
    if not files: raise RuntimeError(f"No tabular file found in {dataset_root}")
    print("Kaggle cache:", dataset_root)
    print("Candidate files:", [str(path.relative_to(dataset_root)) for path in files])
    tasks = {}
    for path in files:
        frame = _read_tabular(path)
        target = _infer_target(frame)
        if target is None:
            print("Skipping file without numeric target:", path.name, list(frame.columns)); continue
        grouped = frame.groupby(TASK_COLUMN, sort=True) if TASK_COLUMN and TASK_COLUMN in frame.columns else [(path.stem, frame)]
        for task_name, group in grouped:
            group = group.dropna(subset=[target]).copy()
            if len(group) < MIN_ROWS_PER_TASK: continue
            excluded = set(DROP_COLUMNS + [target] + ([TASK_COLUMN] if TASK_COLUMN else []))
            numeric_features = group[[c for c in group.columns if c not in excluded]].select_dtypes(include=np.number).columns.tolist()
            if not numeric_features: continue
            x = group[numeric_features].replace([np.inf, -np.inf], np.nan).to_numpy(dtype="float32")
            y = pd.to_numeric(group[target], errors="coerce").to_numpy(dtype="float32").reshape(-1, 1)
            valid = np.isfinite(y).reshape(-1) & np.isfinite(x).all(axis=1)
            x, y = x[valid], y[valid]
            if len(x) < MIN_ROWS_PER_TASK: continue
            if len(x) > MAX_ROWS_PER_TASK:
                selected = np.sort(np.random.default_rng(SEED).choice(len(x), MAX_ROWS_PER_TASK, replace=False))
                x, y = x[selected], y[selected]
            tasks[str(task_name)] = (x, y)
    if not tasks: raise RuntimeError("No eligible regression task found. Set TARGET_COLUMN explicitly and inspect printed columns.")
    return tasks, dataset_root

def split_task(x, y, seed):
    rng = np.random.default_rng(seed)
    index = rng.permutation(len(x))
    lengths = [int(.65 * len(x)), int(.10 * len(x)), int(.15 * len(x))]
    lengths.append(len(x) - sum(lengths))
    boundaries = np.cumsum([0] + lengths)
    return [(x[index[boundaries[j]:boundaries[j + 1]]], y[index[boundaries[j]:boundaries[j + 1]]]) for j in range(4)]

class GaussianMLP(nn.Module):
    def __init__(self, n_features, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU())
        self.mean_head = nn.Linear(hidden, 1)
        self.logvar_head = nn.Linear(hidden, 1)
    def forward(self, x):
        h = self.backbone(x)
        return h, self.mean_head(h), self.logvar_head(h).clamp(-8.0, 5.0)

def train_frozen_model(x_train, y_train, x_val, y_val, seed, epochs=120, patience=15):
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GaussianMLP(x_train.shape[1]).to(device)
    gpu_ids = list(range(torch.cuda.device_count())) if device.type == "cuda" else []
    if len(gpu_ids) >= 2:
        model = nn.DataParallel(model, device_ids=gpu_ids)
        print(f"Using DataParallel on GPUs {gpu_ids}")
    loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)), batch_size=512, shuffle=True, pin_memory=device.type == "cuda")
    val_x_tensor, val_y_tensor = torch.from_numpy(x_val).to(device), torch.from_numpy(y_val).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    best_loss, best_state, stale = float("inf"), None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            _, mean, logvar = model(xb)
            loss = 0.5 * (logvar + (yb - mean).pow(2) * torch.exp(-logvar)).mean()
            optim.zero_grad(set_to_none=True); loss.backward(); optim.step()
        model.eval()
        with torch.inference_mode():
            _, val_mean, val_logvar = model(val_x_tensor)
            val_loss = float((0.5 * (val_logvar + (val_y_tensor - val_mean).pow(2) * torch.exp(-val_logvar))).mean())
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience: break
    if best_state is None: raise RuntimeError("No valid model checkpoint was recorded")
    model.load_state_dict(best_state); model.eval()
    for parameter in model.parameters(): parameter.requires_grad_(False)
    return model, device, best_loss

tasks, KAGGLE_ROOT = load_kaggle_tasks()
print("Loaded Kaggle tasks (no synthetic fallback):", list(tasks), "total:", len(tasks))


## 6. One-pass frozen features, NRC geometry, and calibration

For every task, the penultimate feature matrix and mean-head weight are passed to the published NRC equations. The proposed `D_i` drives a covariance-only post-hoc map fit on calibration data. The test set is untouched until evaluation.


In [ ]:
import time
from geometry.nrc import compute_nrc
from models.adapters import prediction_from_heads
from calibration.nrc_cal import fit_nrc_calibrator
from metrics.evaluation import evaluate
from calibration.baselines import fit_quantile_recalibrator

def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def predict_artifact(model, device, x, y_mean, y_std):
    with torch.inference_mode():
        h, mu, logvar = model(torch.from_numpy(x).to(device, non_blocking=True))
    mu = mu.cpu().numpy() * y_std + y_mean
    variance = np.exp(logvar.cpu().numpy()) * (y_std ** 2)
    return h.cpu().numpy(), prediction_from_heads(mu, np.log(np.maximum(variance, 1e-8)))

def pce_from_pit(pit, levels=100):
    alpha = np.linspace(1 / levels, 1.0, levels)
    return float(np.mean(np.abs((pit[:, None] <= alpha).mean(axis=0) - alpha)))

def fit_feature_geometry(features, mean_head_weight):
    h = np.asarray(features, dtype=float); w = np.asarray(mean_head_weight, dtype=float)
    target_dimension = min(w.shape[0], h.shape[1])
    _, singular_values, right_vectors = np.linalg.svd(h, full_matrices=False)
    pca_basis = right_vectors[:target_dimension].T
    weight_basis, _, _ = np.linalg.svd(w.T, full_matrices=False)
    return {"pca_basis": pca_basis, "weight_basis": weight_basis[:, :target_dimension], "nrc1_singular_values": singular_values}

def score_features(geometry, features, nrc3=0.0, weights=(0.5, 0.5, 0.0)):
    unit = np.asarray(features, dtype=float) / np.linalg.norm(features, axis=1, keepdims=True).clip(min=1e-12)
    def residual(basis):
        projection = (unit @ basis) @ basis.T; delta = unit - projection
        return np.einsum("ij,ij->i", delta, delta)
    r1, r2 = residual(geometry["pca_basis"]), residual(geometry["weight_basis"])
    return weights[0] * r1 + weights[1] * r2 + weights[2] * nrc3, r1, r2

rows, artifacts = [], {}
for task_index, (name, (x, y)) in enumerate(tasks.items()):
    (train_x_raw, train_y_raw), (val_x_raw, val_y_raw), (cal_x_raw, cal_y_raw), (test_x_raw, test_y_raw) = split_task(x, y, SEED + task_index)
    x_median = np.nanmedian(train_x_raw, axis=0)
    clean = lambda values: np.nan_to_num(values, nan=x_median, posinf=x_median, neginf=x_median).astype("float32")
    train_x_raw, val_x_raw, cal_x_raw, test_x_raw = map(clean, (train_x_raw, val_x_raw, cal_x_raw, test_x_raw))
    x_scaler, y_scaler = StandardScaler(), StandardScaler()
    train_x = x_scaler.fit_transform(train_x_raw).astype("float32")
    val_x, cal_x, test_x = [x_scaler.transform(v).astype("float32") for v in (val_x_raw, cal_x_raw, test_x_raw)]
    train_y = y_scaler.fit_transform(train_y_raw).astype("float32")
    val_y, cal_y, test_y = [y_scaler.transform(v).astype("float32") for v in (val_y_raw, cal_y_raw, test_y_raw)]
    model, device, best_val_nll = train_frozen_model(train_x, train_y, val_x, val_y, SEED + task_index)
    base_model = unwrap(model)
    y_mean, y_std = float(y_scaler.mean_[0]), float(y_scaler.scale_[0])
    cal_features, cal_pred = predict_artifact(model, device, cal_x, y_mean, y_std)
    test_features, test_pred = predict_artifact(model, device, test_x, y_mean, y_std)
    cal_targets, test_targets = cal_y * y_std + y_mean, test_y * y_std + y_mean
    mean_head_weight = (y_std * base_model.mean_head.weight.detach().cpu().numpy()).astype(float)
    calibration_nrc = compute_nrc(cal_features, cal_targets, mean_head_weight, weights=(0.5, 0.5, 0.0))
    geometry = fit_feature_geometry(cal_features, mean_head_weight)
    test_distance, _, _ = score_features(geometry, test_features, calibration_nrc.nrc3, calibration_nrc.weights)
    calibrator = fit_nrc_calibrator(cal_pred, cal_targets, calibration_nrc.sample_distance, ridge=1e-6)
    nrc_pred = calibrator.transform(test_pred, test_distance)
    base_metrics, nrc_metrics = evaluate(test_pred, test_targets), evaluate(nrc_pred, test_targets)
    qr = fit_quantile_recalibrator(cal_pred, cal_targets)
    qr_pce = pce_from_pit(qr.map(test_pred.cdf_1d(test_targets)))
    row = {"dataset": name, "nrc_distance": calibration_nrc.dataset_distance, "nrc1": calibration_nrc.nrc1, "nrc2": calibration_nrc.nrc2, "nrc3": calibration_nrc.nrc3, "base_pce": base_metrics.get("pce", np.nan), "qr_pce": qr_pce, "nrc_cal_pce": nrc_metrics.get("pce", np.nan), "base_nll": base_metrics["nll"], "nrc_cal_nll": nrc_metrics["nll"], "base_crps": base_metrics.get("crps", np.nan), "nrc_cal_crps": nrc_metrics.get("crps", np.nan), "base_coverage90": base_metrics.get("coverage_90", np.nan), "nrc_cal_coverage90": nrc_metrics.get("coverage_90", np.nan), "best_validation_nll": best_val_nll}
    rows.append(row); artifacts[name] = {"calibration_nrc": calibration_nrc, "test_distances": test_distance, "base": base_metrics, "nrc_cal": nrc_metrics, "test_targets": test_targets}
    print(f"{name:24s} D={row['nrc_distance']:.4f} BASE-PCE={row['base_pce']:.4f} QR-PCE={qr_pce:.4f} NRC-Cal-PCE={row['nrc_cal_pce']:.4f} device={device} GPUs={torch.cuda.device_count()}")
pilot = pd.DataFrame(rows)
(PROJECT_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
pilot.to_csv(PROJECT_ROOT / "outputs/kaggle_results.csv", index=False)
print("Saved:", PROJECT_ROOT / "outputs/kaggle_results.csv")


## 7. Pilot GO / EXTEND / NO-GO analysis

This is the proposal's first gate. Spearman is the primary association because no linear relationship is assumed. The correlation after QR is the more demanding calibratability test.


In [ ]:
from metrics.statistics import correlations
from scipy.stats import spearmanr

def report_gate(x, y, label):
    result = correlations(x, y, bootstrap_samples=500, permutations=1000, seed=SEED)
    spearman = next(item for item in result if item.name == "spearman")
    print(f"{label}: rho={spearman.coefficient:.3f}, p={spearman.pvalue:.4f}, 95% bootstrap CI=[{spearman.ci_low:.3f}, {spearman.ci_high:.3f}], permutation p={spearman.permutation_pvalue:.4f}")
    return spearman

base_corr = report_gate(pilot.nrc_distance, pilot.base_pce, "NRC distance vs BASE PCE")
qr_corr = report_gate(pilot.nrc_distance, pilot.qr_pce, "NRC distance vs QR PCE")
if abs(base_corr.coefficient) >= 0.5 and base_corr.pvalue < 0.1:
    decision = "GO: expand to the prespecified QRT-57 analysis."
elif abs(base_corr.coefficient) >= 0.3 or abs(qr_corr.coefficient) >= 0.3:
    decision = "EXTEND: increase the pilot to about 20 datasets before deciding."
else:
    decision = "NO-GO for this mechanism on the current pilot; preserve the negative result and investigate."
print("Decision:", decision)


## 8. Figures and ablation checks


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from geometry.nrc import compute_nrc

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(data=pilot, x="nrc_distance", y="base_pce", ax=axes[0], scatter_kws={"s": 55})
axes[0].set_title("Frozen geometry vs BASE PCE")
sns.regplot(data=pilot, x="nrc_distance", y="qr_pce", ax=axes[1], scatter_kws={"s": 55}, color="darkorange")
axes[1].set_title("Frozen geometry vs QR PIT PCE")
fig.tight_layout(); fig.savefig(PROJECT_ROOT / "figures" / "pilot_correlations.png", dpi=180); plt.show()

# Component ablation: the univariate NRC3 component is not applicable by theory.
for component, weights in {"NRC1-only": (1., 0., 0.), "NRC2-only": (0., 1., 0.), "full-univariate": (0.5, 0.5, 0.)}.items():
    values = []
    for artifact in artifacts.values():
        r = artifact["calibration_nrc"]
        values.append(weights[0] * r.nrc1 + weights[1] * r.nrc2 + weights[2] * r.nrc3)
    print(component, "mean distance:", float(np.mean(values)))
print("NRC3-only: not applicable for n=1 because the published NRC definition is identically zero.")


## 9. Optional QRT-57 extension and artifact contract

Use this section only after acquiring real frozen BASE checkpoints. The required per-dataset artifacts are:

`calibration_features.npz`, `calibration_predictions.npz`, `test_features.npz`, `test_predictions.npz`, and `mean_head_weight.npy`. Prediction caches must contain `weights`, `means`, `covariances`, and `targets`. The notebook's embedded `metrics.experiment.run_frozen_nrc_cal` then evaluates BASE and NRC-Cal without changing the checkpoint.

The public QRT repositories expose training/evaluation code but no ready-made checkpoint zoo. If you elect to generate checkpoints, do so as a separate, explicitly documented training stage and retain the exact seed, split, model family, and commit hash.


In [ ]:
# Discover genuine frozen artifacts if you mount them under checkpoints/.
from pathlib import Path
from metrics.experiment import run_frozen_nrc_cal

artifact_root = PROJECT_ROOT / "checkpoints"
checkpoint_files = [p for p in artifact_root.rglob("*") if p.suffix.lower() in {".ckpt", ".pt", ".pth", ".safetensors"}]
print("Checkpoint files found:", len(checkpoint_files))
print("Pilot remains the only automatically executed experiment until the artifact contract is satisfied.")
# Example (adapt only after inspecting the upstream checkpoint architecture):
# result = run_frozen_nrc_cal("Yacht", "BASE-Gaussian", calibration_features, calibration_predictions, test_features, test_predictions, mean_head_weight, output_csv)


## 10. Export and interpretation

The pilot artifacts are under `outputs/` and `figures/`. Interpret the gate as evidence about this frozen-model pilot only. A GO result authorizes the next experiment; it is not a claim that NRC-Cal improves calibration on all 57 QRT datasets. A NO-GO result is still scientifically useful if the split, checkpoint provenance, and statistical procedure are preserved.


In [ ]:
export = pilot.sort_values("base_pce").copy()
export["decision"] = decision
export.to_csv(PROJECT_ROOT / "outputs" / "pilot_results_with_decision.csv", index=False)
print(export[["dataset", "nrc_distance", "base_pce", "qr_pce", "nrc_cal_pce"]].to_string(index=False))
print("\nArtifacts:")
print(" -", PROJECT_ROOT / "outputs" / "pilot_results_with_decision.csv")
print(" -", PROJECT_ROOT / "figures" / "pilot_correlations.png")
print(" -", PROJECT_ROOT / "outputs" / "environment.json")
